# LDA analysis of 65 publications

Reproduction notebook accompanying Appendix A. Run cells deliberately in order. The research profile performs 208 fits for the recorded configuration; no computation has been run in this notebook. Human interpretation remains provisional. Qwen is outside this notebook.


In [ ]:
from pathlib import Path
import sys, json, subprocess
WORK = Path.cwd() / "lda_reproduction"
WORK.mkdir(exist_ok=True)
SOURCE = '"""Reproducible PDF-to-LDA analysis for the 65-publication metaverse/cryptocurrency study.\n\nUse the accompanying README.md, run_config.json and requirements-resolved.txt.\nThe analytical implementation is pipeline version 1.0.1.\n"""\nfrom __future__ import annotations\n\nimport os\n# Colab sets an inline notebook backend in the parent process. This standalone\n# analysis environment renders files, so select Agg BEFORE importing matplotlib.\nos.environ[\'MPLBACKEND\'] = \'Agg\'\nfor _key in (\'OMP_NUM_THREADS\', \'OPENBLAS_NUM_THREADS\', \'MKL_NUM_THREADS\'):\n    os.environ.setdefault(_key, \'1\')\nos.environ.setdefault(\'MPLCONFIGDIR\', os.path.join(os.getcwd(), \'work\', \'matplotlib_cache\'))\nimport argparse\nimport csv\nimport hashlib\nimport importlib.metadata as im\nimport itertools\nimport json\nimport math\nimport platform\nimport re\nimport sys\nimport time\nimport unicodedata\nimport warnings\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom dataclasses import asdict, dataclass, field, replace\nfrom pathlib import Path\n\nimport joblib\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport networkx as nx\nimport numpy as np\nimport pandas as pd\nimport pymupdf\nfrom gensim.corpora import Dictionary\nfrom gensim.models import CoherenceModel, Phrases\nfrom nltk.stem import WordNetLemmatizer\nfrom pypdf import PdfReader\nfrom scipy import sparse\nfrom scipy.optimize import linear_sum_assignment\nfrom scipy.spatial.distance import cdist\nfrom sklearn.decomposition import LatentDirichletAllocation\nfrom sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer\nfrom sklearn.metrics import adjusted_rand_score, cohen_kappa_score\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.metrics.pairwise import cosine_similarity\nfrom threadpoolctl import threadpool_limits\n\nVERSION = \'1.0.1\'\nREVIEW_URL = \'tables/reviewer_response_crosswalk.csv\'\nPACKAGES = [\'pymupdf\',\'pypdf\',\'numpy\',\'scipy\',\'pandas\',\'scikit-learn\',\'gensim\',\n            \'matplotlib\',\'networkx\',\'nltk\',\'pyLDAvis\',\'joblib\',\'threadpoolctl\']\n\n\n@dataclass\nclass Config:\n    input_dir: str = \'/content/pdfs\'\n    output_dir: str = \'/content/lda_results\'\n    profile: str = \'research\'  # smoke tests plumbing; research supports reporting.\n    k_values: tuple = tuple(range(2, 13))\n    seeds: tuple = (11, 42, 97, 2026, 31415)\n    folds: int = 3\n    split_seed: int = 20260915\n    max_iter: int = 250             # outer EM updates, with a bound trace at every update\n    max_doc_update_iter: int = 250  # separate inner document inference cap\n    perp_tol: float = 0.1           # absolute successive training perplexity difference\n    mean_change_tol: float = 0.001\n    alpha: float | None = None      # None = 1/K; total document prior concentration = 1\n    eta: float = 0.01\n    unit: str = \'chunk\'             # chunk or document; original PDFs define validation groups\n    chunk_size: int = 400           # normalized tokens; no overlap; never crosses PDFs\n    min_tail: int = 100             # merge a shorter final tail into preceding chunk\n    max_train_chunks_per_doc: int | None = 50  # evenly spaced, without replacement\n    min_document_tokens: int = 100\n    min_df: int = 2                 # counts ORIGINAL PDFs, not chunks\n    max_df: float = 1.0\n    max_features: int = 10000\n    learned_phrases: bool = True\n    phrase_min_count: int = 10\n    phrase_threshold: float = 10.0\n    lemmatize: bool = True          # WordNet noun lookup, no blind stemming\n    strip_references: bool = True\n    reference_min_fraction: float = 0.40\n    near_duplicate_threshold: float = 0.93\n    coherence_topn: int = 10\n    coherence_tolerance: float = 0.02\n    minimum_stability: float = 0.70  # declared heuristic, not a universal validity threshold\n    minimum_diversity: float = 0.60\n    minimum_converged_fraction: float = 0.80\n    final_k_override: int | None = None\n    final_k_reason: str = \'\'\n    bootstrap_replicates: int = 2000\n    network_terms: int = 40\n    network_min_docs: int = 3\n    network_min_jaccard: float = 0.05\n    network_min_npmi: float = 0.05\n    network_max_df: float = 0.90    # ubiquitous terms carry little document-level association information\n    network_max_edges: int = 150\n    run_sensitivity: bool = True\n    enable_ocr: bool = False\n    ocr_language: str = \'eng\'\n    metadata_csv: str = \'\'\n    page_rules_csv: str = \'\'\n    extra_stopwords: tuple = ()\n\n    def checked(self):\n        if self.profile == \'smoke\':\n            self.k_values = (2, 7, 8)\n            self.seeds = (11, 42)\n            self.folds = 2\n            self.max_iter = 12\n            self.max_doc_update_iter = 60\n            self.bootstrap_replicates = 200\n        if self.unit not in (\'document\',\'chunk\') or self.folds < 2 or len(self.seeds) < 2:\n            raise ValueError(\'Use document/chunk, at least two folds and two seeds.\')\n        if len(set(self.seeds)) != len(self.seeds) or len(set(self.k_values)) != len(self.k_values):\n            raise ValueError(\'Seeds and K values must be unique.\')\n        if any(k < 2 for k in self.k_values) or self.chunk_size < self.min_tail:\n            raise ValueError(\'K must be >=2 and chunk_size >= min_tail.\')\n        if self.final_k_override is not None and (self.final_k_override not in self.k_values or not self.final_k_reason.strip()):\n            raise ValueError(\'A final K override must be evaluated in k_values and have a written reason.\')\n        return self\n\n\n# Fixed lexical equivalents are declared BEFORE any fitting. They do not prescribe topics.\n# Ambiguous abbreviations (ML, TF, IP, AR) are intentionally not expanded globally.\nDOMAIN_ALIASES = {\n    \'non fungible tokens\':\'nft\', \'non fungible token\':\'nft\',\n    \'nonfungible tokens\':\'nft\', \'nonfungible token\':\'nft\', \'nfts\':\'nft\',\n    \'cryptocurrencies\':\'cryptocurrency\', \'crypto currencies\':\'cryptocurrency\',\n    \'crypto currency\':\'cryptocurrency\', \'cryptoassets\':\'cryptoasset\',\n    \'crypto assets\':\'cryptoasset\', \'crypto asset\':\'cryptoasset\',\n    \'block chain\':\'blockchain\', \'blockchains\':\'blockchain\', \'metaverses\':\'metaverse\',\n    \'smart contracts\':\'smart_contract\', \'smart contract\':\'smart_contract\',\n    \'decentralized finance\':\'defi\', \'decentralised finance\':\'defi\',\n    \'decentralized autonomous organizations\':\'dao\', \'decentralised autonomous organisations\':\'dao\',\n    \'decentralized autonomous organization\':\'dao\', \'decentralised autonomous organisation\':\'dao\', \'daos\':\'dao\',\n    \'virtual reality\':\'virtual_reality\', \'augmented reality\':\'augmented_reality\',\n    \'mixed reality\':\'mixed_reality\', \'extended reality\':\'extended_reality\',\n    \'artificial intelligence\':\'artificial_intelligence\', \'machine learning\':\'machine_learning\',\n    \'deep learning\':\'deep_learning\', \'generative ai\':\'generative_ai\',\n    \'federated learning\':\'federated_learning\', \'digital twins\':\'digital_twin\', \'digital twin\':\'digital_twin\',\n    \'digital assets\':\'digital_asset\', \'digital asset\':\'digital_asset\',\n    \'digital identities\':\'digital_identity\', \'digital identity\':\'digital_identity\',\n    \'identity theft\':\'identity_theft\', \'identity impersonation\':\'identity_impersonation\',\n    \'access control\':\'access_control\', \'attribute based encryption\':\'attribute_based_encryption\',\n    \'self sovereign identity\':\'self_sovereign_identity\', \'self sovereign ai\':\'self_sovereign_ai\',\n    \'zero knowledge proofs\':\'zero_knowledge_proof\', \'zero knowledge proof\':\'zero_knowledge_proof\',\n    \'post quantum\':\'post_quantum\', \'quantum computing\':\'quantum_computing\',\n    \'money laundering\':\'money_laundering\', \'terrorist financing\':\'terrorist_financing\',\n    \'terrorism financing\':\'terrorist_financing\', \'financial crime\':\'financial_crime\',\n    \'financial crimes\':\'financial_crime\', \'financial fraud\':\'financial_fraud\',\n    \'wash trading\':\'wash_trading\', \'market manipulation\':\'market_manipulation\',\n    \'insider trading\':\'insider_trading\', \'tax evasion\':\'tax_evasion\',\n    \'rug pulls\':\'rug_pull\', \'rug pull\':\'rug_pull\', \'ponzi schemes\':\'ponzi_scheme\', \'ponzi scheme\':\'ponzi_scheme\',\n    \'social engineering\':\'social_engineering\', \'consumer protection\':\'consumer_protection\',\n    \'consumer rights\':\'consumer_right\', \'intellectual property\':\'intellectual_property\',\n    \'data protection\':\'data_protection\', \'data privacy\':\'data_privacy\',\n    \'virtual worlds\':\'virtual_world\', \'virtual world\':\'virtual_world\',\n    \'real estate\':\'real_estate\', \'atomic swaps\':\'atomic_swap\', \'atomic swap\':\'atomic_swap\',\n    \'cross chain\':\'cross_chain\', \'cross metaverse\':\'cross_metaverse\',\n    \'proof of personhood\':\'proof_of_personhood\', \'sybil attacks\':\'sybil_attack\', \'sybil attack\':\'sybil_attack\',\n    \'cyber security\':\'cybersecurity\', \'cyber crimes\':\'cybercrime\', \'cyber crime\':\'cybercrime\',\n    \'cybercrimes\':\'cybercrime\', \'deep fakes\':\'deepfake\', \'deepfakes\':\'deepfake\',\n    \'web 3.0\':\'web3\', \'web 3\':\'web3\', \'decentralised\':\'decentralized\',\n    \'normalised\':\'normalized\', \'behaviour\':\'behavior\', \'behaviours\':\'behavior\',\n}\nMODEL_BOILERPLATE = {\'doi\',\'isbn\',\'issn\',\'copyright\',\'sciencedirect\',\'elsevier\',\n    \'springer\',\'ieee\',\'et\',\'al\',\'fig\',\'figure\',\'figures\',\'table\',\'tables\',\'vol\',\n    \'pp\',\'www\',\'http\',\'https\',\'com\',\'org\',\'edu\',\'author\',\'authors\',\'rights\',\'reserved\',\n    \'abstract\',\'keywords\',\'references\',\'bibliography\',\'received\',\'accepted\',\'published\'}\nDISPLAY_STOPWORDS = {\'paper\',\'study\',\'research\',\'analysis\',\'result\',\'based\',\'using\',\'use\',\n    \'approach\',\'model\',\'method\',\'proposed\',\'propose\',\'proposes\',\'new\',\'different\',\'also\',\n    \'metaverse\',\'blockchain\',\'technology\',\'system\',\'application\',\'include\',\'provide\',\n    \'used\',\'can\',\'may\',\'will\',\'one\',\'two\',\'et\',\'al\',\'article\',\'work\',\'appendix\',\'section\',\n    \'issue\',\'potential\',\'like\',\'various\',\'however\',\'therefore\',\'time\',\'address\',\'challenge\',\n    \'framework\',\'future\',\'development\',\'environment\',\'mention\',\'process\',\'information\',\n    \'digital\',\'virtual\',\'blockchain_technology\',\'metaverse_technology\'}\n\n\ndef write_json(path, obj):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    def convert(x):\n        if isinstance(x, (np.integer,np.floating)): return x.item()\n        if isinstance(x, np.ndarray): return x.tolist()\n        if isinstance(x, Path): return str(x)\n        raise TypeError(type(x).__name__)\n    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=convert), encoding=\'utf-8\')\n\n\ndef table(rows, path, columns=None):\n    df = rows if isinstance(rows,pd.DataFrame) else pd.DataFrame(rows,columns=columns)\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    df.to_csv(path,index=False,encoding=\'utf-8-sig\')\n    return df\n\n\ndef slug(text):\n    return re.sub(r\'[^a-zA-Z0-9_-]+\',\'_\',str(text)).strip(\'_\')[:100]\n\n\ndef digest(obj):\n    return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()\n\n\ndef save_plot(fig, directory, name):\n    directory=Path(directory); directory.mkdir(parents=True,exist_ok=True)\n    fig.tight_layout()\n    fig.savefig(directory/(name+\'.png\'),dpi=300,bbox_inches=\'tight\')\n    fig.savefig(directory/(name+\'.svg\'),bbox_inches=\'tight\')\n    fig.savefig(directory/(name+\'.pdf\'),bbox_inches=\'tight\')\n    plt.close(fig)\n\n\ndef setup_resources(out, cfg):\n    import nltk\n    path=Path(out)/\'private\'/\'nltk_data\'; path.mkdir(parents=True,exist_ok=True)\n    nltk.data.path.insert(0,str(path))\n    if cfg.lemmatize:\n        try: WordNetLemmatizer().lemmatize(\'wallets\')\n        except LookupError:\n            if not nltk.download(\'wordnet\',download_dir=str(path),quiet=True,raise_on_error=True):\n                raise RuntimeError(\'WordNet download failed. Restore network access, or explicitly set lemmatize=False and document the change.\')\n            WordNetLemmatizer().lemmatize(\'wallets\')\n        from nltk.corpus import wordnet\n        write_json(Path(out)/\'tables\'/\'nlp_resource_version.json\',{\'wordnet\':wordnet.get_version(),\n            \'lemmatizer\':\'WordNetLemmatizer; noun POS; domain aliases applied first\'})\n\n\ndef clean_unicode(text):\n    text=unicodedata.normalize(\'NFKC\',text).replace(\'\\u00ad\',\'\')\n    text=re.sub(r\'(?<=[A-Za-z])-\\s*\\n\\s*(?=[a-z])\',\'\',text)\n    return text\n\n\ndef extract_pdf(path, cfg):\n    """Local extraction. Block order preserves within-paragraph phrase context.\n    PyMuPDF restores word spacing in the supplied MetaRepo PDF; pypdf is fallback.\n    OCR is optional and logged, never silently treated as identical extraction.\n    """\n    pages=[]; audits=[]\n    try:\n        with pymupdf.open(path) as pdf:\n            if pdf.needs_pass: raise ValueError(\'Password-protected PDF\')\n            for index,page in enumerate(pdf):\n                text=\'\\n\'.join(b[4] for b in page.get_text(\'blocks\',sort=True) if b[6]==0)\n                method=\'pymupdf_blocks\'\n                words=re.findall(r\'[A-Za-z]+\',text)\n                image_area=sum(pymupdf.Rect(x[\'bbox\']).get_area() for x in page.get_image_info())\n                image_fraction=min(1.0,image_area/max(1,page.rect.get_area()))\n                probable_scan=len(words)<30 and image_fraction>0.65\n                if probable_scan and cfg.enable_ocr:\n                    tp=page.get_textpage_ocr(language=cfg.ocr_language,dpi=300,full=True)\n                    text=page.get_text(textpage=tp); method=\'tesseract_ocr\'\n                text=clean_unicode(text); pages.append(text)\n                audits.append({\'page\':index+1,\'method\':method,\'characters\':len(text),\n                    \'word_count\':len(text.split()),\'image_fraction\':image_fraction,\n                    \'probable_scan\':probable_scan,\'low_text\':len(words)<30})\n    except Exception as first_error:\n        if cfg.enable_ocr: raise RuntimeError(f\'Extraction/OCR failed for {path.name}: {first_error}\') from first_error\n        reader=PdfReader(path)\n        pages=[clean_unicode(p.extract_text() or \'\') for p in reader.pages]\n        audits=[{\'page\':i+1,\'method\':\'pypdf_fallback\',\'characters\':len(t),\'word_count\':len(t.split()),\n                 \'fallback_reason\':str(first_error),\'low_text\':len(t.split())<30} for i,t in enumerate(pages)]\n    return pages,audits\n\n\ndef strip_pages(pages, cfg, rule=None):\n    """Return (physical page number, cleaned text) with a complete action log."""\n    rule=rule or {}; actions=[]\n    start=int(rule.get(\'first_page\') or 1); end=int(rule.get(\'last_page\') or len(pages))\n    if not 1<=start<=end<=len(pages): raise ValueError(f\'Invalid physical page range {start}:{end}\')\n    retained=[]\n    for i,t in enumerate(pages,1):\n        if not start<=i<=end:\n            actions.append({\'page\':i,\'action\':\'page_rule_exclusion\',\'reason\':rule.get(\'reason\',\'\')}); continue\n        if sum(s in t.lower() for s in (\'rapid #\',\'lender:\',\'borrower:\',\'cross ref id\'))>=2:\n            actions.append({\'page\':i,\'action\':\'library_delivery_cover\'}); continue\n        retained.append((i,t))\n    candidates=Counter()\n    for _,t in retained:\n        lines=[x.strip() for x in t.splitlines() if x.strip()]\n        candidates.update(set(x for x in lines[:2]+lines[-2:] if len(x)<150))\n    repeated={x for x,n in candidates.items() if n>=max(3,math.ceil(len(retained)*.4))}\n    result=[]\n    for p,t in retained:\n        lines=[]\n        for line in t.splitlines():\n            line=line.strip()\n            if line in repeated or re.fullmatch(r\'\\d{1,5}\',line or \' \'):\n                if line: actions.append({\'page\':p,\'action\':\'repeated_margin_or_page_number\',\'text\':line})\n            else: lines.append(line)\n        result.append((p,\'\\n\'.join(lines)))\n    total=sum(len(t) for _,t in result); consumed=0; stopped=False; trimmed=[]\n    for p,t in result:\n        keep=[]\n        for line in t.splitlines():\n            heading=re.sub(r\'^[\\dIVXivx.\\s]+\',\'\',line.strip())\n            heading=re.sub(r\'\\s+\',\'\',heading).lower().rstrip(\':\')\n            if cfg.strip_references and heading in (\'references\',\'bibliography\',\'workscited\') and consumed/max(total,1)>=cfg.reference_min_fraction:\n                actions.append({\'page\':p,\'action\':\'reference_cut\',\'heading\':line,\n                    \'fraction_of_clean_text\':consumed/max(total,1)}); stopped=True; break\n            keep.append(line); consumed+=len(line)+1\n        trimmed.append((p,\'\\n\'.join(keep)))\n        if stopped: break\n    actions.append({\'action\':\'reference_status\',\'removed\':stopped,\n                    \'rule\':f\'standalone heading after {cfg.reference_min_fraction:.0%} of text\'})\n    return trimmed,actions\n\n\ndef normalize_pages(pages, cfg):\n    stops=set(ENGLISH_STOP_WORDS)|MODEL_BOILERPLATE|set(cfg.extra_stopwords)\n    aliases=sorted(DOMAIN_ALIASES,key=len,reverse=True)\n    pattern=re.compile(r\'(?<![a-z0-9_])(\'+ \'|\'.join(re.escape(x) for x in aliases)+r\')(?![a-z0-9_])\')\n    lemma=WordNetLemmatizer(); audit=Counter(); output=[]\n    for page,text in pages:\n        t=clean_unicode(text).lower()\n        t=re.sub(r\'https?://\\S+|www\\.\\S+|\\b\\S+@\\S+\\b\',\' \',t)\n        t=re.sub(r\'10\\.\\d{4,9}/\\S+\',\' \',t)\n        t=re.sub(r\'[\\u2010-\\u2015-]\',\' \',t)\n        t=re.sub(r\'[^a-z0-9_\\s.]\',\' \',t); t=re.sub(r\'\\s+\',\' \',t)\n        def sub(m):\n            src=m.group(); dest=DOMAIN_ALIASES[src]\n            audit[(src,dest,\'domain_alias\')]+=1\n            return dest\n        t=pattern.sub(sub,t)\n        tokens=[]; previous=None\n        for token in re.findall(r\'[a-z][a-z0-9_]*\',t):\n            if len(token)<3 or len(token)>35 or token in stops: continue\n            canonical=lemma.lemmatize(token,pos=\'n\') if cfg.lemmatize and \'_\' not in token else token\n            if canonical!=token: audit[(token,canonical,\'wordnet_noun\')]+=1\n            if canonical in stops: continue\n            if canonical==previous:\n                audit[(token,canonical,\'adjacent_duplicate_removed\')]+=1; continue\n            tokens.append(canonical); previous=canonical\n        output.append((page,tokens))\n    return output,audit\n\n\ndef load_page_rules(cfg):\n    # Exact content hashes make the packaged override safe if a PDF is renamed.\n    builtin=globals().get(\'CORPUS_PAGE_RULES\',{})\n    rules=dict(builtin)\n    if cfg.page_rules_csv:\n        for row in pd.read_csv(cfg.page_rules_csv).fillna(\'\').to_dict(\'records\'):\n            key=row.get(\'sha256\') or row.get(\'relative_path\')\n            if key: rules[str(key)]=row\n    return rules\n\n\ndef audit_corpus(cfg, out):\n    source=Path(cfg.input_dir).expanduser().resolve()\n    files=sorted((p for p in source.rglob(\'*\') if p.is_file() and p.suffix.lower()==\'.pdf\'),key=lambda p:p.relative_to(source).as_posix().lower())\n    if not files: raise FileNotFoundError(f\'No PDFs in {source}. Upload/extract the corpus or set the Drive folder.\')\n    docs=[]; records=[]; page_rows=[]; cleaning=[]; normalization=Counter(); seen_sha={}; seen_body={}; failures=[]\n    rules=load_page_rules(cfg)\n    for i,path in enumerate(files,1):\n        relative=path.relative_to(source).as_posix(); sha=hashlib.sha256(path.read_bytes()).hexdigest()\n        doc_id=\'D\'+sha[:12]; row={\'document_id\':doc_id,\'relative_path\':relative,\'filename\':path.name,\'sha256\':sha,\'status\':\'pending\'}\n        if sha in seen_sha:\n            row.update(status=\'exact_file_duplicate\',duplicate_of=seen_sha[sha]); records.append(row); continue\n        seen_sha[sha]=doc_id\n        try:\n            raw,pages_audit=extract_pdf(path,cfg)\n            for r in pages_audit: page_rows.append(dict(document_id=doc_id,**r))\n            rule=rules.get(sha,rules.get(relative,{}))\n            clean,actions=strip_pages(raw,cfg,rule)\n            for action in actions: cleaning.append(dict(document_id=doc_id,**action))\n            body=\'\\n\'.join(t for _,t in clean)\n            body_hash=hashlib.sha256(re.sub(r\'\\s+\',\'\',body.lower()).encode()).hexdigest()\n            norm,audit=normalize_pages(clean,cfg)\n            tokens=[w for _,ws in norm for w in ws]\n            row.update(page_count=len(raw),raw_words=sum(len(t.split()) for t in raw),\n                cleaned_words=len(body.split()),normalized_tokens=len(tokens),body_hash=body_hash,\n                reference_removed=any(x[\'action\']==\'reference_cut\' for x in actions),\n                extraction_review_flag=any(x.get(\'probable_scan\') for x in pages_audit),\n                long_word_fraction=float(np.mean([len(x)>25 for x in body.split()])) if body.split() else 1.0)\n            if len(tokens)<cfg.min_document_tokens:\n                raise ValueError(f\'Only {len(tokens)} usable tokens: check extraction/OCR; no silent exclusion\')\n            if body_hash in seen_body:\n                row.update(status=\'exact_body_duplicate\',duplicate_of=seen_body[body_hash]); records.append(row); continue\n            seen_body[body_hash]=doc_id\n            row[\'status\']=\'included\'; records.append(row); normalization.update(audit)\n            docs.append({\'id\':doc_id,\'filename\':path.name,\'relative_path\':relative,\'sha256\':sha,\n                         \'pages\':clean,\'normalized_pages\':norm,\'tokens\':tokens,\'text\':body,\'group\':doc_id})\n            private=out/\'private\'/\'extracted\'; private.mkdir(parents=True,exist_ok=True)\n            write_json(private/(doc_id+\'.json\'),{\'raw_pages\':raw,\'cleaned_pages\':clean,\'normalization\':norm})\n        except Exception as e:\n            row.update(status=\'error\',error=str(e)); records.append(row); failures.append(row)\n        print(f\'PDF {i}/{len(files)}: {path.name}: {row["status"]}\',flush=True)\n    manifest=table(records,out/\'tables\'/\'corpus_manifest.csv\')\n    table(page_rows,out/\'tables\'/\'extraction_page_audit.csv\')\n    table(cleaning,out/\'private\'/\'cleaning_actions.csv\')\n    table([dict(original_form=a,normalized_form=b,mapping_type=c,count=n) for (a,b,c),n in normalization.items()],out/\'tables\'/\'normalization_map.csv\')\n    write_json(out/\'tables\'/\'corpus_flow.json\',{\'files_discovered\':len(files),\'included\':len(docs),\n        \'exact_file_duplicates\':sum(x[\'status\']==\'exact_file_duplicate\' for x in records),\n        \'exact_body_duplicates\':sum(x[\'status\']==\'exact_body_duplicate\' for x in records),\n        \'errors\':len(failures),\'screening_prisma\':\'Not reconstructed from this folder; use the separate screening record.\'})\n    if failures: raise RuntimeError(\'Extraction failed; see corpus_manifest.csv. Fix the listed PDFs before rerunning.\')\n    if len(docs)<max(12,2*max(cfg.k_values)):\n        warnings.warn(f\'Only {len(docs)} articles for K up to {max(cfg.k_values)}. Exploratory results may be weak.\')\n    # Near matches are retained and grouped for validation, not silently deduplicated.\n    tf=TfidfVectorizer(analyzer=\'char_wb\',ngram_range=(5,5),min_df=2,max_features=40000).fit_transform([d[\'text\'] for d in docs])\n    sims=cosine_similarity(tf); parent=list(range(len(docs)))\n    def find(x):\n        while parent[x]!=x: parent[x]=parent[parent[x]]; x=parent[x]\n        return x\n    pairs=[]\n    for a,b in itertools.combinations(range(len(docs)),2):\n        if sims[a,b]>=cfg.near_duplicate_threshold:\n            pairs.append({\'document_a\':docs[a][\'id\'],\'document_b\':docs[b][\'id\'],\'cosine_char5\':sims[a,b],\n                          \'decision\':\'retained; same validation group; human publication-identity check pending\'})\n            parent[find(b)]=find(a)\n    for i,d in enumerate(docs): d[\'group\']=docs[find(i)][\'id\']\n    table(pairs,out/\'tables\'/\'near_duplicate_candidates.csv\',columns=[\'document_a\',\'document_b\',\'cosine_char5\',\'decision\'])\n    table([{\'document_id\':d[\'id\'],\'filename\':d[\'filename\'],\'publication_type\':\'unknown\',\'type_verified\':False,\n        \'year\':\'\',\'screening_id\':\'\',\'quality_appraisal_tool\':\'\',\'quality_appraisal_result\':\'\',\'notes\':\'\'} for d in docs],out/\'tables\'/\'metadata_template.csv\')\n    return docs,manifest\n\n\ndef fit_representation(docs,cfg):\n    # Only training PDFs are passed here during cross-validation.\n    sentences=[ws for d in docs for _,ws in d[\'normalized_pages\'] if ws]\n    phraser=Phrases(sentences,min_count=cfg.phrase_min_count,threshold=cfg.phrase_threshold,delimiter=\'_\').freeze() if cfg.learned_phrases else None\n    texts=[[w for _,ws in d[\'normalized_pages\'] for w in (phraser[ws] if phraser else ws)] for d in docs]\n    dictionary=Dictionary(texts)\n    dictionary.filter_extremes(no_below=cfg.min_df,no_above=cfg.max_df,keep_n=cfg.max_features)\n    dictionary.compactify()\n    if len(dictionary)<max(30,2*max(cfg.k_values)): raise ValueError(\'Vocabulary too small. Inspect preprocessing and document-frequency limits.\')\n    return {\'phraser\':phraser,\'dictionary\':dictionary}\n\n\ndef chunk_spans(n,size,min_tail):\n    if n==0: return []\n    spans=[(i,min(i+size,n)) for i in range(0,n,size)]\n    if len(spans)>1 and spans[-1][1]-spans[-1][0]<min_tail:\n        spans[-2]=(spans[-2][0],spans[-1][1]); spans.pop()\n    return spans\n\n\ndef make_units(docs,rep,cfg):\n    dictionary=rep[\'dictionary\']; phraser=rep[\'phraser\']\n    units=[]; bows=[]; texts=[]; coverage=[]\n    for d in docs:\n        words=[]; locations=[]\n        for p,ws in d[\'normalized_pages\']:\n            transformed=list(phraser[ws]) if phraser else ws\n            words.extend(transformed); locations.extend([p]*len(transformed))\n        texts.append([w for w in words if w in dictionary.token2id])\n        known=sum(w in dictionary.token2id for w in words)\n        coverage.append({\'document_id\':d[\'id\'],\'tokens_before_vocabulary\':len(words),\'in_vocabulary_tokens\':known,\n                         \'oov_fraction\':1-known/max(1,len(words))})\n        spans=[(0,len(words))] if cfg.unit==\'document\' else chunk_spans(len(words),cfg.chunk_size,cfg.min_tail)\n        for j,(start,end) in enumerate(spans):\n            bow=dictionary.doc2bow(words[start:end])\n            if not bow: raise ValueError(f\'Empty vocabulary unit in {d["filename"]}. Adjust preprocessing; no silent drop.\')\n            units.append({\'document_id\':d[\'id\'],\'unit_index\':j,\'start_token\':start,\'end_token\':end,\n                \'page_start\':locations[start],\'page_end\':locations[end-1],\'token_count\':sum(v for _,v in bow),\n                \'normalized_excerpt\':\' \'.join(words[start:min(end,start+180)])})\n            bows.append(bow)\n    rr=[]; cc=[]; vv=[]\n    for i,bow in enumerate(bows):\n        for j,v in bow: rr.append(i); cc.append(j); vv.append(v)\n    X=sparse.csr_matrix((vv,(rr,cc)),shape=(len(bows),len(dictionary)),dtype=np.float64)\n    selected=[]\n    for d in docs:\n        ix=[i for i,u in enumerate(units) if u[\'document_id\']==d[\'id\']]\n        cap=cfg.max_train_chunks_per_doc\n        if cfg.unit==\'chunk\' and cap is not None and len(ix)>cap:\n            ix=[ix[k] for k in np.linspace(0,len(ix)-1,cap,dtype=int)]\n        selected.extend(ix)\n    return {\'X\':X,\'units\':units,\'training_indices\':np.array(selected,dtype=int),\'texts\':texts,\'coverage\':coverage}\n\n\nclass TracedLDA(LatentDirichletAllocation):\n    """Observe sklearn 1.7.2\'s own training evaluations, without changing updates.\n    Its private hook is version checked and tested; no score(X) calls are added\n    inside EM. n_iter_ has a stopping-edge convention, so count EM calls directly.\n    """\n    def fit(self,X,y=None):\n        if im.version(\'scikit-learn\')!=\'1.7.2\':\n            raise RuntimeError(\'Install scikit-learn==1.7.2 for the tested convergence-trace hook.\')\n        self.trace_=[]; self.actual_iterations_=0; self._record_trace=True\n        try: super().fit(X,y)\n        finally: self._record_trace=False\n        self.converged_=len(self.trace_)>=2 and abs(self.trace_[-1][\'perplexity\']-self.trace_[-2][\'perplexity\'])<self.perp_tol\n        return self\n\n    def _em_step(self,X,total_samples,batch_update,parallel=None):\n        self.actual_iterations_+=1\n        return super()._em_step(X,total_samples,batch_update,parallel)\n\n    def _perplexity_precomp_distr(self,X,doc_topic_distr=None,sub_sampling=False):\n        value=super()._perplexity_precomp_distr(X,doc_topic_distr,sub_sampling)\n        if getattr(self,\'_record_trace\',False):\n            row={\'iteration\':self.actual_iterations_,\'perplexity\':float(value),\n                 \'variational_bound\':float(-np.log(value)*X.sum()),\'bound_per_token\':float(-np.log(value))}\n            if self.trace_ and self.trace_[-1][\'iteration\']==self.actual_iterations_: self.trace_[-1]=row\n            else: self.trace_.append(row)\n        return value\n\n\ndef fit_model(X,k,seed,cfg,cache):\n    cache=Path(cache)\n    if cache.exists():\n        try: return joblib.load(cache)\n        except Exception as e:\n            print(f\'Unreadable checkpoint {cache.name}; refitting ({type(e).__name__}).\',flush=True)\n    model=TracedLDA(n_components=k,doc_topic_prior=cfg.alpha,topic_word_prior=cfg.eta,\n        learning_method=\'batch\',max_iter=cfg.max_iter,max_doc_update_iter=cfg.max_doc_update_iter,\n        evaluate_every=1,perp_tol=cfg.perp_tol,mean_change_tol=cfg.mean_change_tol,n_jobs=1,random_state=int(seed))\n    before=time.time()\n    with threadpool_limits(limits=1): model.fit(X)\n    model.elapsed_seconds_=time.time()-before\n    cache.parent.mkdir(parents=True,exist_ok=True)\n    temporary=cache.with_name(cache.name+\'.tmp\')\n    joblib.dump(model,temporary)\n    temporary.replace(cache)  # promote only a fully written model\n    return model\n\n\ndef phi(model):\n    return model.components_/model.components_.sum(axis=1,keepdims=True)\n\n\ndef top_terms(model,dictionary,n=10):\n    return [[dictionary[int(i)] for i in row.argsort()[::-1][:n]] for row in phi(model)]\n\n\ndef aggregate_documents(theta,units,ids,weighted=True):\n    answer=[]\n    for doc_id in ids:\n        mask=np.array([u[\'document_id\']==doc_id for u in units])\n        weights=np.array([u[\'token_count\'] for u in units])[mask] if weighted else None\n        answer.append(np.average(theta[mask],axis=0,weights=weights))\n    a=np.array(answer); return a/a.sum(axis=1,keepdims=True)\n\n\ndef match_topics(A,B,topn=10):\n    # sqrt(JS divergence base 2) is in [0,1]; similarity is one minus distance.\n    distance=cdist(A,B,metric=lambda x,y: float(np.sqrt(max(0,0.5*np.sum(x*np.log2(x/((x+y)/2)))+0.5*np.sum(y*np.log2(y/((x+y)/2)))))))\n    rows,cols=linear_sum_assignment(distance)\n    jacc=[]\n    for a,b in zip(rows,cols):\n        sa=set(A[a].argsort()[-topn:]); sb=set(B[b].argsort()[-topn:]); jacc.append(len(sa&sb)/len(sa|sb))\n    return rows,cols,1-distance[rows,cols],np.array(jacc)\n\n\ndef seed_stability(models,data,doc_ids):\n    phis=[phi(m) for m in models]\n    doc_theta=[aggregate_documents(m.transform(data[\'X\']),data[\'units\'],doc_ids) for m in models]\n    pairs=[]; sums=np.zeros(len(models))\n    for a,b in itertools.combinations(range(len(models)),2):\n        rows,cols,js,jacc=match_topics(phis[a],phis[b]); score=float(js.mean()); sums[a]+=score; sums[b]+=score\n        aligned=doc_theta[b][:,cols]\n        pairs.append({\'seed_a\':models[a].random_state,\'seed_b\':models[b].random_state,\n            \'matched_js_similarity\':score,\'matched_top10_jaccard\':float(jacc.mean()),\n            \'dominant_topic_ari\':adjusted_rand_score(doc_theta[a].argmax(1),doc_theta[b].argmax(1)),\n            \'document_topic_mean_l1\':float(np.abs(doc_theta[a][:,rows]-aligned).sum(1).mean())})\n    medoid=int(np.argmax(sums)) # deterministic order resolves a tie; never choose maximum coherence seed\n    return pairs,medoid\n\n\ndef split_completion(X,seed):\n    rng=np.random.default_rng(seed); observed=X.copy(); target=X.copy()\n    observed.data=rng.binomial(X.data.astype(np.int64),.5).astype(float)\n    target.data=X.data-observed.data; observed.eliminate_zeros(); target.eliminate_zeros()\n    # Zero halves are possible for very short units; fail visibly instead of leaking held-out words.\n    if np.any(np.asarray(observed.sum(1)).ravel()==0) or np.any(np.asarray(target.sum(1)).ravel()==0):\n        raise ValueError(\'A completion half is empty. Increase chunk size or inspect very sparse documents.\')\n    assert (observed+target-X).nnz==0\n    return observed,target\n\n\ndef completion_scores(model,observed,target,unigram,units):\n    theta=model.transform(observed); topics=phi(model); rows=[]\n    for i in range(target.shape[0]):\n        row=target.getrow(i); prob=theta[i]@topics[:,row.indices]\n        ll=float(row.data@np.log(np.maximum(prob,1e-300)))\n        baseline=float(row.data@np.log(unigram[row.indices]))\n        rows.append({\'document_id\':units[i][\'document_id\'],\'unit_index\':units[i][\'unit_index\'],\n                     \'completion_log_likelihood\':ll,\'unigram_log_likelihood\':baseline,\'heldout_tokens\':float(row.sum())})\n    docs=pd.DataFrame(rows).groupby(\'document_id\',as_index=False)[[\'completion_log_likelihood\',\'unigram_log_likelihood\',\'heldout_tokens\']].sum()\n    docs[\'log_likelihood_per_token\']=docs.completion_log_likelihood/docs.heldout_tokens\n    docs[\'unigram_per_token\']=docs.unigram_log_likelihood/docs.heldout_tokens\n    docs[\'gain_vs_unigram_nats\']=docs.log_likelihood_per_token-docs.unigram_per_token\n    summary={\'validation_macro_loglik_per_token\':float(docs.log_likelihood_per_token.mean()),\n        \'validation_completion_perplexity\':float(np.exp(-docs.log_likelihood_per_token.mean())),\n        \'validation_total_completion_loglik\':float(docs.completion_log_likelihood.sum()),\n        \'validation_micro_loglik_per_token\':float(docs.completion_log_likelihood.sum()/docs.heldout_tokens.sum()),\n        \'validation_gain_vs_unigram_nats\':float(docs.gain_vs_unigram_nats.mean())}\n    return summary,docs\n\n\ndef batch_coherence(models,texts,dictionary,topn):\n    lists=[top_terms(m,dictionary,topn) for m in models]; scores=[{} for _ in models]\n    for metric,window in [(\'c_v\',110),(\'c_npmi\',10)]:\n        cm=CoherenceModel.for_topics(lists,texts=texts,dictionary=dictionary,coherence=metric,window_size=window,processes=1,topn=topn)\n        for result,topics in zip(scores,lists):\n            cm.topics=topics\n            values=np.array(cm.get_coherence_per_topic(),dtype=float)\n            result[metric]=float(np.mean(values)); result[metric+\'_per_topic\']=values.tolist()\n    for result,topics in zip(scores,lists):\n        result[\'topic_diversity\']=len(set(itertools.chain.from_iterable(topics)))/sum(map(len,topics))\n    return scores\n\n\n# Verified physical PDF pages: the first seven pages are delivery/book front matter.\n# The filename says Cryptocurrency; the actual chapter title says Cryptography.\nCORPUS_PAGE_RULES = {\n    \'f993b1e11c98058f5746d549a0dfd6f94ec92340d5b178bc024f2aacbf2f3029\': {\n        \'first_page\':8,\'last_page\':50,\n        \'reason\':\'Chapter 14, Cryptography in the metaverse: advanced protocols for secure communication; remove delivery sheet and book front matter.\'}\n}\n\n\ndef cross_validate(docs,cfg,out):\n    groups=[d[\'group\'] for d in docs]\n    if len(set(groups))<cfg.folds: raise ValueError(\'Too few independent document groups for requested folds.\')\n    splitter=GroupKFold(n_splits=cfg.folds,shuffle=True,random_state=cfg.split_seed)\n    all_rows=[]; all_stability=[]; splits=[]; all_scores=[]; coverage=[]; traces=[]\n    for fold,(tr,va) in enumerate(splitter.split(np.arange(len(docs)),groups=groups),1):\n        train=[docs[i] for i in tr]; valid=[docs[i] for i in va]\n        assert not ({d[\'group\'] for d in train}&{d[\'group\'] for d in valid})\n        for part,subset in [(\'train\',train),(\'validation\',valid)]:\n            splits.extend({\'fold\':fold,\'part\':part,\'document_id\':d[\'id\'],\'group\':d[\'group\']} for d in subset)\n        rep=fit_representation(train,cfg); td=make_units(train,rep,cfg); vd=make_units(valid,rep,cfg)\n        trainX=td[\'X\'][td[\'training_indices\']]\n        observed,target=split_completion(vd[\'X\'],cfg.split_seed+fold)\n        unigram=np.asarray(trainX.sum(0)).ravel()+.5; unigram/=unigram.sum()\n        coverage.extend(dict(fold=fold,**x) for x in vd[\'coverage\'])\n        vocabulary=rep[\'dictionary\']\n        write_json(out/\'private\'/f\'fold_{fold}_vocabulary.json\',vocabulary.token2id)\n        fold_models=[]; fold_rows=[]; models_by_k={}\n        for k in cfg.k_values:\n            models=[]\n            for seed in cfg.seeds:\n                print(f\'CV fold {fold}/{cfg.folds} | K={k} | seed={seed}\',flush=True)\n                model=fit_model(trainX,k,seed,cfg,out/\'private\'/\'cache\'/f\'cv_f{fold}_k{k}_s{seed}.joblib\')\n                score,perdoc=completion_scores(model,observed,target,unigram,vd[\'units\'])\n                perdoc[\'fold\']=fold; perdoc[\'k\']=k; perdoc[\'seed\']=seed\n                all_scores.extend(perdoc.to_dict(\'records\'))\n                row=dict(fold=fold,k=k,seed=seed,train_documents=len(train),validation_documents=len(valid),\n                    train_units=trainX.shape[0],vocabulary_size=trainX.shape[1],\n                    alpha=model.doc_topic_prior_,eta=model.topic_word_prior_,\n                    train_variational_bound=model.trace_[-1][\'variational_bound\'],\n                    train_bound_per_token=model.trace_[-1][\'bound_per_token\'],\n                    train_bound_perplexity=model.trace_[-1][\'perplexity\'],\n                    iterations=model.actual_iterations_,converged=model.converged_,seconds=model.elapsed_seconds_,**score)\n                fold_models.append(model); fold_rows.append(row); models.append(model)\n                traces.extend(dict(fold=fold,k=k,seed=seed,**t) for t in model.trace_)\n            pairs,_=seed_stability(models,td,[d[\'id\'] for d in train])\n            all_stability.extend(dict(fold=fold,k=k,**p) for p in pairs)\n            models_by_k[k]=models\n        print(f\'CV fold {fold}: computing cached C_v and C_NPMI co-occurrences\',flush=True)\n        coherence=batch_coherence(fold_models,td[\'texts\'],vocabulary,cfg.coherence_topn)\n        for row,coh in zip(fold_rows,coherence):\n            row.update({key:value for key,value in coh.items() if not key.endswith(\'_per_topic\')})\n        all_rows.extend(fold_rows)\n        table(all_rows,out/\'tables\'/\'model_selection_runs.csv\')\n        table(all_stability,out/\'tables\'/\'seed_stability_cv.csv\')\n        table(traces,out/\'tables\'/\'convergence_traces_cv.csv\')\n    table(splits,out/\'tables\'/\'document_validation_splits.csv\')\n    table(all_scores,out/\'tables\'/\'validation_document_completion.csv\')\n    table(coverage,out/\'tables\'/\'validation_vocabulary_coverage.csv\')\n    runs=pd.DataFrame(all_rows); stability=pd.DataFrame(all_stability)\n    summary=[]\n    for k,g in runs.groupby(\'k\'):\n        row={\'k\':int(k),\'runs\':len(g),\'converged_fraction\':float(g.converged.mean())}\n        for col in [\'c_v\',\'c_npmi\',\'topic_diversity\',\'validation_macro_loglik_per_token\',\n                    \'validation_gain_vs_unigram_nats\',\'train_bound_per_token\',\'train_bound_perplexity\']:\n            fold_means=g.groupby(\'fold\')[col].mean()\n            row[col+\'_mean\']=float(fold_means.mean())\n            row[col+\'_fold_sd\']=float(fold_means.std(ddof=1))\n            row[col+\'_fold_se\']=float(fold_means.std(ddof=1)/np.sqrt(len(fold_means)))\n            row[col+\'_mean_seed_sd\']=float(g.groupby(\'fold\')[col].std(ddof=1).mean())\n        sg=stability[stability.k==k]\n        for col in [\'matched_js_similarity\',\'matched_top10_jaccard\',\'dominant_topic_ari\',\'document_topic_mean_l1\']:\n            row[col+\'_mean\']=float(sg[col].mean()); row[col+\'_pair_sd\']=float(sg[col].std(ddof=1))\n        row[\'validation_completion_perplexity\']=float(np.exp(-row[\'validation_macro_loglik_per_token_mean\']))\n        summary.append(row)\n    summary=table(summary,out/\'tables\'/\'model_selection_summary.csv\')\n    return summary,runs\n\n\ndef recommend_k(summary,cfg,out):\n    s=summary.copy()\n    needed=[\'c_v_mean\',\'validation_macro_loglik_per_token_mean\',\'matched_js_similarity_mean\']\n    if not np.isfinite(s[needed].to_numpy()).all():\n        raise ValueError(\'Non-finite selection metrics. Inspect vocabulary/coherence and diagnostic runs.\')\n    best=s.loc[s.c_v_mean.idxmax()]\n    tolerance=max(cfg.coherence_tolerance,float(best.c_v_fold_se))\n    s[\'coherence_shortlist\']=s.c_v_mean>=float(best.c_v_mean)-tolerance\n    s[\'stability_gate\']=s.matched_js_similarity_mean>=cfg.minimum_stability\n    s[\'diversity_gate\']=s.topic_diversity_mean>=cfg.minimum_diversity\n    s[\'convergence_gate\']=s.converged_fraction>=cfg.minimum_converged_fraction\n    eligible=s[s.coherence_shortlist&s.stability_gate&s.diversity_gate&s.convergence_gate]\n    gates_passed=not eligible.empty\n    if eligible.empty: eligible=s[s.coherence_shortlist]\n    predictive=eligible.loc[eligible.validation_macro_loglik_per_token_mean.idxmax()]\n    cutoff=float(predictive.validation_macro_loglik_per_token_mean-predictive.validation_macro_loglik_per_token_fold_se)\n    candidates=eligible[eligible.validation_macro_loglik_per_token_mean>=cutoff]\n    recommended=int(candidates.k.min())\n    chosen=cfg.final_k_override or recommended\n    note=[]\n    if not gates_passed: note.append(\'No coherence-shortlisted K passed all declared stability, diversity, and convergence gates. Recommendation is provisional; inspect and rerun before reporting.\')\n    if recommended in (min(cfg.k_values),max(cfg.k_values)): note.append(\'Recommendation lies at the tested K boundary; expand the range before claiming a resolved optimum.\')\n    if cfg.profile!=\'research\': note.append(\'Smoke profile: validation only, not publication evidence or a substantive K recommendation.\')\n    note.append(\'Human interpretability checks by at least two coders remain pending until completed rating files are supplied.\')\n    decision={\'recommended_k\':recommended,\'final_k\':int(chosen),\'tested_k\':list(cfg.k_values),\n        \'gates_passed\':gates_passed,\'coherence_best_k\':int(best.k),\'coherence_tolerance_used\':tolerance,\n        \'predictive_cutoff_nats_per_token\':cutoff,\'manual_override_reason\':cfg.final_k_reason,\n        \'rule\':\'Shortlist within max(0.02, best-K fold SE) of best mean C_v; apply declared stability/diversity/convergence gates; retain candidates within one fold SE of best validation completion log score; choose smallest K. If gates fail, retain coherence shortlist and mark provisional.\',\n        \'uncertainty_note\':\'Fold SE and dependent seed-pair SD are descriptive model-selection aids, not independent inferential confidence intervals.\',\n        \'notes\':note}\n    table(s,out/\'tables\'/\'model_selection_decision_table.csv\')\n    write_json(out/\'model_selection_decision.json\',decision)\n    fig,axes=plt.subplots(2,3,figsize=(15,8))\n    specs=[(\'c_v_mean\',\'c_v_fold_sd\',\'C_v coherence (fold SD)\'),\n           (\'validation_completion_perplexity\',None,\'Validation completion perplexity\'),\n           (\'matched_js_similarity_mean\',\'matched_js_similarity_pair_sd\',\'Matched topic stability (dependent pair SD)\'),\n           (\'validation_macro_loglik_per_token_mean\',\'validation_macro_loglik_per_token_fold_sd\',\'Validation log score per token (fold SD)\'),\n           (\'topic_diversity_mean\',None,\'Top-10 topic diversity\'),(\'converged_fraction\',None,\'Converged runs / all runs\')]\n    for ax,(col,err,title) in zip(axes.flat,specs):\n        ax.errorbar(s.k,s[col],yerr=s[err] if err else None,marker=\'o\',capsize=3,color=\'#126782\')\n        ax.axvline(chosen,color=\'#c95e35\',linestyle=\'--\',label=f\'K={chosen}\')\n        ax.set(title=title,xlabel=\'Number of topics (K)\'); ax.set_xticks(s.k); ax.grid(alpha=.2)\n    fig.suptitle(\'Model comparison before final fitting — \'+(\'SMOKE TEST\' if cfg.profile==\'smoke\' else \'document-grouped validation\'))\n    save_plot(fig,out/\'figures\',\'model_selection_diagnostics\')\n    return decision\n\n\ndef fit_final_candidates(docs,cfg,out,decision):\n    rep=fit_representation(docs,cfg); data=make_units(docs,rep,cfg)\n    chosen=decision[\'final_k\']\n    candidates=sorted({k for k in (chosen-1,chosen,chosen+1,7,8) if k in cfg.k_values})\n    families={}; flat=[]; rows=[]; pair_rows=[]; traces=[]\n    X=data[\'X\'][data[\'training_indices\']]\n    for k in candidates:\n        models=[]\n        for seed in cfg.seeds:\n            print(f\'Final corpus | K={k} | seed={seed}\',flush=True)\n            m=fit_model(X,k,seed,cfg,out/\'private\'/\'cache\'/f\'full_k{k}_s{seed}.joblib\')\n            models.append(m); flat.append(m)\n            rows.append({\'k\':k,\'seed\':seed,\'iterations\':m.actual_iterations_,\'converged\':m.converged_,\n                         \'bound_per_token\':m.trace_[-1][\'bound_per_token\'],\'perplexity_bound\':m.trace_[-1][\'perplexity\']})\n            traces.extend(dict(k=k,seed=seed,**r) for r in m.trace_)\n        pairs,medoid=seed_stability(models,data,[d[\'id\'] for d in docs])\n        pair_rows.extend(dict(k=k,**r) for r in pairs)\n        families[k]={\'models\':models,\'medoid\':medoid,\'model\':models[medoid]}\n    cohs=batch_coherence(flat,data[\'texts\'],rep[\'dictionary\'],cfg.coherence_topn)\n    for row,coh in zip(rows,cohs):\n        row.update({k:v for k,v in coh.items() if not k.endswith(\'_per_topic\')})\n    for m,coh in zip(flat,cohs): m.topic_coherence_=coh\n    table(rows,out/\'tables\'/\'full_corpus_seed_diagnostics.csv\')\n    table(pair_rows,out/\'tables\'/\'full_corpus_seed_stability.csv\')\n    table(traces,out/\'tables\'/\'convergence_traces_full_corpus.csv\')\n    table(data[\'coverage\'],out/\'tables\'/\'full_corpus_vocabulary_coverage.csv\')\n    table([{k:v for k,v in u.items() if k!=\'normalized_excerpt\'} for u in data[\'units\']],out/\'tables\'/\'unit_manifest.csv\')\n    table(data[\'units\'],out/\'private\'/\'unit_manifest_with_passages.csv\')\n    rep[\'dictionary\'].save(str(out/\'private\'/\'dictionary.gensim\'))\n    if rep[\'phraser\']: rep[\'phraser\'].save(str(out/\'private\'/\'phrases.gensim\'))\n    sparse.save_npz(out/\'private\'/\'unit_term_counts.npz\',data[\'X\'])\n    table([{\'token_id\':i,\'term\':rep[\'dictionary\'][i],\n        \'document_frequency\':rep[\'dictionary\'].dfs.get(i,0)} for i in range(len(rep[\'dictionary\']))],out/\'tables\'/\'vocabulary.csv\')\n    phrase_counts=Counter(itertools.chain.from_iterable(data[\'texts\']))\n    phrase_rows=[{\'phrase\':term,\'association_score\':score,\'corpus_occurrences\':phrase_counts[term]}\n        for term,score in rep[\'phraser\'].phrasegrams.items()] if rep[\'phraser\'] else []\n    table(phrase_rows,out/\'tables\'/\'learned_phrases.csv\',columns=[\'phrase\',\'association_score\',\'corpus_occurrences\'])\n    model=families[chosen][\'model\']\n    joblib.dump(model,out/\'private\'/\'final_lda_model.joblib\')\n    fig,axes=plt.subplots(1,2,figsize=(12,4))\n    for m in families[chosen][\'models\']:\n        trace=pd.DataFrame(m.trace_)\n        axes[0].plot(trace.iteration,trace.bound_per_token,label=f\'seed {m.random_state}\')\n        axes[1].plot(trace.iteration,trace.perplexity,label=f\'seed {m.random_state}\')\n    axes[0].set(title=\'Training variational lower bound\',xlabel=\'Actual outer EM update\',ylabel=\'Bound per token (nats)\')\n    axes[1].set(title=\'Training bound-based perplexity\',xlabel=\'Actual outer EM update\',ylabel=\'Perplexity\')\n    for ax in axes: ax.legend(fontsize=8); ax.grid(alpha=.2)\n    save_plot(fig,out/\'figures\',\'final_convergence_trace\')\n    return rep,data,families\n\n\ndef topic_word_tables(model,rep,prevalence,cfg):\n    P=phi(model); dictionary=rep[\'dictionary\']; background=prevalence@P\n    summaries=[]; terms=[]\n    for t,prob in enumerate(P):\n        raw=prob.argsort()[::-1]\n        pool=[int(i) for i in raw[:100] if dictionary[int(i)] not in DISPLAY_STOPWORDS]\n        def score(i):\n            term=dictionary[i]\n            return .6*np.log(prob[i])+.4*np.log(prob[i]/background[i])+(.10 if \'_\' in term else 0)\n        chosen=sorted(pool,key=lambda i:(-score(i),dictionary[i]))[:10]\n        selected_mass=sum(prob[i] for i in chosen)\n        label=\' / \'.join(dictionary[i].replace(\'_\',\' \') for i in chosen[:3])\n        summaries.append({\'topic\':f\'T{t+1:02d}\',\'suggested_label\':label,\'label_status\':\'provisional; human interpretation pending\',\n            \'raw_top10\':\', \'.join(dictionary[int(i)] for i in raw[:10]),\n            \'display_top10\':\', \'.join(dictionary[i] for i in chosen),\'selected_probability_mass\':selected_mass,\n            \'mean_document_prevalence\':prevalence[t],\n            \'c_v\':model.topic_coherence_[\'c_v_per_topic\'][t],\n            \'c_npmi\':model.topic_coherence_[\'c_npmi_per_topic\'][t]})\n        chosen_order={i:j+1 for j,i in enumerate(chosen)}\n        for rank,i in enumerate(raw[:100],1):\n            i=int(i)\n            terms.append({\'topic\':f\'T{t+1:02d}\',\'term\':dictionary[i],\'raw_rank\':rank,\n                \'p_word_given_topic\':prob[i],\'exclusivity\':prob[i]/P[:,i].sum(),\n                \'lift_vs_corpus\':prob[i]/background[i],\n                \'display_rank\':chosen_order.get(i,np.nan),\n                \'normalized_weight_within_display_top10\':prob[i]/selected_mass if i in chosen_order else np.nan,\n                \'display_score\':score(i),\'display_hidden\':dictionary[i] in DISPLAY_STOPWORDS})\n    return pd.DataFrame(summaries),pd.DataFrame(terms)\n\n\ndef export_topics(docs,rep,data,families,cfg,out,decision):\n    family=families[decision[\'final_k\']]; model=family[\'model\']; P=phi(model); k=P.shape[0]\n    ids=[d[\'id\'] for d in docs]; topic_ids=[f\'T{i+1:02d}\' for i in range(k)]\n    unit_theta=model.transform(data[\'X\'])\n    theta=aggregate_documents(unit_theta,data[\'units\'],ids,weighted=True)\n    equal_chunk=aggregate_documents(unit_theta,data[\'units\'],ids,weighted=False)\n    prev=theta.mean(0)\n    assert np.allclose(theta.sum(1),1) and np.isclose(prev.sum(),1)\n    dt=pd.DataFrame(theta,columns=topic_ids); dt.insert(0,\'document_id\',ids); dt.insert(1,\'filename\',[d[\'filename\'] for d in docs])\n    table(dt,out/\'tables\'/\'document_topic_matrix.csv\')\n    entropy=-(theta*np.log(np.maximum(theta,1e-300))).sum(1)\n    table([{\'document_id\':ids[i],\'dominant_topic\':topic_ids[int(theta[i].argmax())],\n        \'dominant_topic_weight\':float(theta[i].max()),\'normalized_topic_entropy\':float(entropy[i]/np.log(k)),\n        \'effective_topic_count\':float(np.exp(entropy[i]))} for i in range(len(docs))],out/\'tables\'/\'document_topic_diagnostics.csv\')\n    ut=pd.DataFrame(unit_theta,columns=topic_ids)\n    ut.insert(0,\'document_id\',[u[\'document_id\'] for u in data[\'units\']]); ut.insert(1,\'unit_index\',[u[\'unit_index\'] for u in data[\'units\']])\n    table(ut,out/\'tables\'/\'unit_topic_matrix.csv\')\n    summary,keywords=topic_word_tables(model,rep,prev,cfg)\n    table(keywords,out/\'tables\'/\'topic_keywords_extended.csv\')\n    table(keywords[keywords.display_rank.notna()].sort_values([\'topic\',\'display_rank\']),out/\'tables\'/\'topic_top10_normalized_keywords.csv\')\n    rows=[]; private_passages=[]; counts=[]\n    chosen_train=set(data[\'training_indices\'])\n    for d in docs:\n        ix=[i for i,u in enumerate(data[\'units\']) if u[\'document_id\']==d[\'id\']]\n        counts.append({\'document_id\':d[\'id\'],\'filename\':d[\'filename\'],\'all_units\':len(ix),\n            \'training_units\':sum(i in chosen_train for i in ix),\'fewer_than_50_units\':len(ix)<50,\n            \'inference_uses_all_units\':True,\'in_vocabulary_tokens\':int(sum(data[\'units\'][i][\'token_count\'] for i in ix))})\n    table(counts,out/\'tables\'/\'document_chunk_counts.csv\')\n    for t in range(k):\n        for rank,i in enumerate(np.argsort(theta[:,t])[::-1][:3],1):\n            doc=docs[i]; ix=[j for j,u in enumerate(data[\'units\']) if u[\'document_id\']==doc[\'id\']]\n            best=max(ix,key=lambda j:unit_theta[j,t]); unit=data[\'units\'][best]\n            row={\'topic\':topic_ids[t],\'rank\':rank,\'document_id\':doc[\'id\'],\'filename\':doc[\'filename\'],\n                 \'document_topic_weight\':theta[i,t],\'representative_unit\':unit[\'unit_index\'],\n                 \'pdf_page_start\':unit[\'page_start\'],\'pdf_page_end\':unit[\'page_end\']}\n            rows.append(row); private_passages.append(dict(**row,normalized_passage=unit[\'normalized_excerpt\']))\n    reps=table(rows,out/\'tables\'/\'representative_documents.csv\')\n    table(private_passages,out/\'private\'/\'representative_passages.csv\')\n    summary[\'representative_documents\']=[\', \'.join(reps[reps.topic==tid].document_id) for tid in topic_ids]\n    table(summary,out/\'tables\'/\'topic_summary.csv\')\n    rng=np.random.default_rng(cfg.split_seed)\n    boot=np.array([theta[rng.integers(0,len(docs),len(docs))].mean(0) for _ in range(cfg.bootstrap_replicates)])\n    seed_prev=[]; matched=[]\n    for other in family[\'models\']:\n        rr,cc,js,jac=match_topics(P,phi(other))\n        other_theta=aggregate_documents(other.transform(data[\'X\']),data[\'units\'],ids)[:,cc]\n        seed_prev.append(other_theta.mean(0))\n        matched.extend({\'reference_topic\':topic_ids[a],\'other_topic\':topic_ids[b],\'seed\':other.random_state,\n                        \'js_similarity\':s,\'top10_jaccard\':j} for a,b,s,j in zip(rr,cc,js,jac))\n    seed_prev=np.array(seed_prev)\n    prevalence=pd.DataFrame({\'topic\':topic_ids,\'mean_document_prevalence\':prev,\'percent\':100*prev,\n        \'bootstrap_low_conditional\':np.quantile(boot,.025,axis=0),\n        \'bootstrap_high_conditional\':np.quantile(boot,.975,axis=0),\n        \'seed_aligned_mean\':seed_prev.mean(0),\'seed_aligned_sd\':seed_prev.std(0,ddof=1),\n        \'seed_aligned_min\':seed_prev.min(0),\'seed_aligned_max\':seed_prev.max(0),\n        \'equal_chunk_within_document_prevalence\':equal_chunk.mean(0),\n        \'naive_all_chunk_prevalence\':unit_theta.mean(0),\n        \'token_weighted_corpus_prevalence\':np.average(unit_theta,axis=0,weights=[u[\'token_count\'] for u in data[\'units\']]),\n        \'dominant_document_count\':np.bincount(theta.argmax(1),minlength=k)})\n    table(prevalence,out/\'tables\'/\'topic_prevalence.csv\'); table(matched,out/\'tables\'/\'seed_topic_alignment.csv\')\n    table([{\'topic\':tid,\'seed\':m.random_state,\'mean_document_prevalence\':seed_prev[s,t]} for s,m in enumerate(family[\'models\']) for t,tid in enumerate(topic_ids)],out/\'tables\'/\'seed_aligned_prevalence.csv\')\n    fig,ax=plt.subplots(figsize=(10,5)); order=np.argsort(prev)\n    lo=prevalence.bootstrap_low_conditional.to_numpy(); hi=prevalence.bootstrap_high_conditional.to_numpy()\n    ax.barh(np.array(topic_ids)[order],prev[order]*100,color=\'#126782\')\n    ax.errorbar(prev[order]*100,np.arange(k),xerr=np.array([np.maximum(0,prev-lo),np.maximum(0,hi-prev)])[:,order]*100,fmt=\'none\',ecolor=\'#202b3c\',capsize=3)\n    ax.set(xlabel=\'Mean topic weight across original PDFs (%)\',title=\'Equal-document topic prevalence\\n95% bootstrap intervals conditional on the fixed fitted model\')\n    save_plot(fig,out/\'figures\',\'average_topic_prevalence\')\n    fig,ax=plt.subplots(figsize=(11,5)); image=ax.imshow(theta,aspect=\'auto\',cmap=\'Blues\',vmin=0,vmax=1)\n    ax.set(xticks=np.arange(k),xticklabels=topic_ids,xlabel=\'Topic\',ylabel=\'Original PDF (manifest order)\',title=\'Document-topic proportions\')\n    fig.colorbar(image,ax=ax,label=\'Topic weight\'); save_plot(fig,out/\'figures\',\'document_topic_heatmap\')\n    fig,ax=plt.subplots(figsize=(8,4)); ax.hist([x[\'all_units\'] for x in counts],bins=min(15,len(docs)),color=\'#126782\',edgecolor=\'white\')\n    ax.axvline(50,color=\'#c95e35\',ls=\'--\',label=\'50-unit training cap; not an eligibility threshold\')\n    ax.set(xlabel=\'Units in an original PDF\',ylabel=\'Number of PDFs\',title=\'Chunk-count distribution\'); ax.legend(fontsize=8)\n    save_plot(fig,out/\'figures\',\'chunk_count_distribution\')\n    # Save complete normalized topic-word matrix, not just a top-term approximation.\n    table(pd.DataFrame(P.T,columns=topic_ids).assign(term=[rep[\'dictionary\'][i] for i in range(P.shape[1])]),out/\'tables\'/\'topic_word_probabilities.csv\')\n    for method in (\'cosine\',\'js\'):\n        S=cosine_similarity(P) if method==\'cosine\' else 1-cdist(P,P,metric=lambda x,y:float(np.sqrt(max(0,.5*np.sum(x*np.log2(x/((x+y)/2)))+.5*np.sum(y*np.log2(y/((x+y)/2)))))))\n        sim=pd.DataFrame(S,columns=topic_ids); sim.insert(0,\'topic\',topic_ids)\n        table(sim,out/\'tables\'/f\'topic_similarity_{method}.csv\')\n        fig,ax=plt.subplots(figsize=(8,7)); imshow=ax.imshow(S,vmin=0,vmax=1,cmap=\'Blues\')\n        ax.set(xticks=np.arange(k),yticks=np.arange(k),xticklabels=topic_ids,yticklabels=topic_ids,title=f\'Topic-word similarity: {method.upper()}\')\n        if k<=15:\n            for a in range(k):\n                for b in range(k): ax.text(b,a,f\'{S[a,b]:.2f}\',ha=\'center\',va=\'center\',fontsize=8,color=\'white\' if S[a,b]>.65 else \'#222222\')\n        fig.colorbar(imshow,ax=ax,label=\'Similarity\'); save_plot(fig,out/\'figures\',f\'topic_similarity_{method}\')\n    quality=[{\'metric\':\'training_variational_lower_bound\',\'value\':model.trace_[-1][\'variational_bound\'],\'interpretation\':\'Approximate evidence lower bound in nats; not exact marginal log likelihood.\'},\n        {\'metric\':\'training_bound_perplexity\',\'value\':model.trace_[-1][\'perplexity\'],\'interpretation\':\'Training bound-derived perplexity, lower is better on the same data/vocabulary.\'},\n        {\'metric\':\'overall_c_v\',\'value\':model.topic_coherence_[\'c_v\'],\'interpretation\':\'Unweighted mean across raw top-10 topic coherences; window 110.\'},\n        {\'metric\':\'overall_c_npmi\',\'value\':model.topic_coherence_[\'c_npmi\'],\'interpretation\':\'Unweighted mean across raw top-10 topic coherences; window 10.\'},\n        {\'metric\':\'topic_diversity_top10\',\'value\':model.topic_coherence_[\'topic_diversity\'],\'interpretation\':\'Unique terms across all top-10 lists divided by 10K.\'},\n        {\'metric\':\'outer_iterations\',\'value\':model.actual_iterations_,\'interpretation\':\'Actual EM updates observed by the trace hook.\'},\n        {\'metric\':\'converged\',\'value\':int(model.converged_),\'interpretation\':\'Successive training perplexity difference below configured tolerance; no guarantee of global optimum.\'},\n        {\'metric\':\'articles\',\'value\':len(docs),\'interpretation\':\'Independent sampling units remain original documents, not chunks.\'}]\n    table(quality,out/\'tables\'/\'final_goodness_of_fit.csv\')\n    try:\n        import pyLDAvis\n        frequencies=np.asarray(data[\'X\'].sum(0)).ravel(); lengths=np.asarray(data[\'X\'].sum(1)).ravel()\n        vis=pyLDAvis.prepare(P,unit_theta,lengths,[rep[\'dictionary\'][i] for i in range(P.shape[1])],frequencies,sort_topics=False,n_jobs=1)\n        pyLDAvis.save_html(vis,str(out/\'lda_interactive.html\'))\n        write_json(out/\'tables\'/\'pyldavis_status.json\',{\'status\':\'created\',\'topic_numbering\':\'1 corresponds to T01; sorting disabled\',\n            \'weighting\':\'pyLDAvis topic areas use token-weighted units; use topic_prevalence.csv for equal-document estimates.\'})\n    except Exception as e:\n        write_json(out/\'tables\'/\'pyldavis_status.json\',{\'status\':\'failed\',\'error\':str(e)})\n        raise RuntimeError(f\'pyLDAvis export failed: {e}\') from e\n    return theta,summary,keywords\n\n\ndef keyword_network(docs,rep,data,cfg,out):\n    """One document contributes at most one count to an edge. No overlapping windows."""\n    ids=[d[\'id\'] for d in docs]\n    Xdoc=sparse.vstack([sparse.csr_matrix(data[\'X\'][[i for i,u in enumerate(data[\'units\']) if u[\'document_id\']==d]].sum(0)) for d in ids],format=\'csr\')\n    counts=np.asarray(Xdoc.sum(0)).ravel(); lengths=np.asarray(Xdoc.sum(1)).ravel()\n    rel=sparse.diags(1/lengths)@Xdoc\n    macro=np.asarray(rel.mean(0)).ravel(); binary=Xdoc.copy(); binary.data[:]=1\n    df=np.asarray(binary.sum(0)).ravel(); words=[rep[\'dictionary\'][i] for i in range(Xdoc.shape[1])]\n    kws=pd.DataFrame({\'term\':words,\'total_count\':counts.astype(int),\'document_frequency\':df.astype(int),\n        \'document_percent\':100*df/len(docs),\'frequency_per_10000_tokens\':10000*counts/counts.sum(),\n        \'mean_document_frequency_per_10000_tokens\':10000*macro})\n    kws=kws.sort_values([\'mean_document_frequency_per_10000_tokens\',\'term\'],ascending=[False,True])\n    table(kws,out/\'tables\'/\'all_normalized_keywords.csv\'); table(kws.head(10),out/\'tables\'/\'top10_normalized_keywords_overall.csv\')\n    meaningful=kws[~kws.term.isin(DISPLAY_STOPWORDS)]\n    table(meaningful.head(10),out/\'tables\'/\'top10_normalized_keywords_display.csv\')\n    fig,ax=plt.subplots(figsize=(10,5)); top=kws.head(10).iloc[::-1]\n    ax.barh(top.term.str.replace(\'_\',\' \'),top.mean_document_frequency_per_10000_tokens,color=\'#126782\')\n    ax.set(xlabel=\'Mean within-document occurrences per 10,000 retained tokens\',title=\'Top 10 normalized keywords — equal document weighting\')\n    save_plot(fig,out/\'figures\',\'top10_normalized_keywords\')\n    candidates=meaningful[(meaningful.document_frequency>=cfg.network_min_docs)&\n        (meaningful.document_frequency<=cfg.network_max_df*len(docs))].head(cfg.network_terms).term.tolist()\n    ix=[rep[\'dictionary\'].token2id[w] for w in candidates]\n    co=(binary[:,ix].T@binary[:,ix]).toarray(); freq=df[ix]; edges=[]\n    for a,b in itertools.combinations(range(len(ix)),2):\n        n=int(co[a,b]); union=freq[a]+freq[b]-n\n        jacc=n/union if union else 0\n        p=n/len(docs); denom=-np.log(p) if p>0 else np.inf\n        npmi=np.log(p/((freq[a]/len(docs))*(freq[b]/len(docs))))/denom if p>0 and denom>0 else 0.0\n        if n>=cfg.network_min_docs and jacc>=cfg.network_min_jaccard and npmi>=cfg.network_min_npmi:\n            edges.append({\'source\':candidates[a],\'target\':candidates[b],\'cooccurring_documents\':n,\n                          \'jaccard\':float(jacc),\'npmi\':float(npmi)})\n    edges=sorted(edges,key=lambda e:(-e[\'jaccard\'],-e[\'cooccurring_documents\'],e[\'source\'],e[\'target\']))\n    table(edges,out/\'tables\'/\'keyword_network_all_eligible_edges.csv\',columns=[\'source\',\'target\',\'cooccurring_documents\',\'jaccard\',\'npmi\'])\n    shown=edges[:cfg.network_max_edges]; G=nx.Graph()\n    for word in candidates:\n        i=rep[\'dictionary\'].token2id[word]\n        G.add_node(word,document_frequency=int(df[i]),normalized_frequency=float(macro[i]*10000))\n    for e in shown: G.add_edge(e[\'source\'],e[\'target\'],weight=e[\'jaccard\'],cooccurring_documents=e[\'cooccurring_documents\'],npmi=e[\'npmi\'])\n    communities=list(nx.community.greedy_modularity_communities(G,weight=\'weight\')) if G.number_of_edges() else [{x} for x in G]\n    cluster={w:c for c,group in enumerate(communities,1) for w in group}\n    nodes=[]\n    for node,attrs in G.nodes(data=True):\n        attrs[\'community\']=cluster[node]; nodes.append({\'term\':node,**attrs,\'degree\':G.degree(node)})\n    table(nodes,out/\'tables\'/\'keyword_network_nodes.csv\'); table(shown,out/\'tables\'/\'keyword_network_display_edges.csv\',columns=[\'source\',\'target\',\'cooccurring_documents\',\'jaccard\',\'npmi\'])\n    nx.write_graphml(G,out/\'keyword_cooccurrence.graphml\')\n    # Circular positions avoid collapsed clusters/overlapping labels when a few\n    # ubiquitous terms form a dense subgraph. Geometry does not imply distance.\n    ordered=sorted(G,key=lambda node:(cluster[node],node))\n    fig,ax=plt.subplots(figsize=(14,14)); pos=nx.circular_layout(ordered)\n    if G:\n        nx.draw_networkx_edges(G,pos,ax=ax,alpha=.22,width=[.5+3*x[\'weight\'] for _,_,x in G.edges(data=True)])\n        nx.draw_networkx_nodes(G,pos,ax=ax,node_size=[100+6*G.nodes[x][\'document_frequency\'] for x in G],node_color=[cluster[x] for x in G],cmap=\'tab20\',alpha=.9)\n        for node,(x,y) in pos.items():\n            angle=np.degrees(np.arctan2(y,x)); left=x<0\n            ax.text(x*1.07,y*1.07,node.replace(\'_\',\' \'),fontsize=9,\n                rotation=angle+(180 if left else 0),rotation_mode=\'anchor\',\n                ha=\'right\' if left else \'left\',va=\'center\')\n    ax.set_title(\'Keyword co-occurrence across original PDFs\\nPositive associations among non-ubiquitous terms; descriptive, not independent LDA validation\',pad=25)\n    ax.set_xlim(-1.65,1.65); ax.set_ylim(-1.65,1.65); ax.axis(\'off\'); save_plot(fig,out/\'figures\',\'keyword_cooccurrence_network\')\n    write_json(out/\'tables\'/\'keyword_network_definition.json\',{\'unit\':\'original PDF; binary term presence\',\n        \'nodes\':\'top document-normalized terms after display-only filtering and maximum document-frequency filter\',\n        \'edge_weight\':\'Jaccard = joint document count / union document count\',\n        \'minimum_cooccurring_documents\':cfg.network_min_docs,\'minimum_jaccard\':cfg.network_min_jaccard,\n        \'minimum_npmi\':cfg.network_min_npmi,\'maximum_document_frequency_fraction\':cfg.network_max_df,\n        \'plot_edge_cap\':cfg.network_max_edges,\'eligible_edges\':len(edges),\'plotted_edges\':len(shown),\n        \'community_method\':\'greedy modularity on displayed graph; not LDA topic assignments\'})\n    return Xdoc\n\n\ndef sensitivity_analysis(docs,rep,data,families,cfg,out,decision,theta):\n    rows=[]; pending=[]; models=[]; ref=families[decision[\'final_k\']][\'model\']; P=phi(ref)\n    k=decision[\'final_k\']; ids=[d[\'id\'] for d in docs]\n    variants=[(\'whole_document_units\',replace(cfg,unit=\'document\'),docs),\n        (\'chunks_200\',replace(cfg,unit=\'chunk\',chunk_size=200,min_tail=50),docs),\n        (\'chunks_800\',replace(cfg,unit=\'chunk\',chunk_size=800,min_tail=100),docs),\n        (\'uncapped_training_chunks\',replace(cfg,max_train_chunks_per_doc=None),docs),\n        (\'alpha_0_1\',replace(cfg,alpha=.1),docs),(\'eta_0_1\',replace(cfg,eta=.1),docs)]\n    metadata=None\n    if cfg.metadata_csv:\n        metadata=pd.read_csv(cfg.metadata_csv).fillna(\'\')\n        required={\'document_id\',\'publication_type\',\'type_verified\'}\n        if not required.issubset(metadata): raise ValueError(f\'Metadata needs {required}\')\n        if metadata.document_id.duplicated().any(): raise ValueError(\'Duplicate document IDs in metadata.\')\n        merged=pd.DataFrame({\'document_id\':ids}).merge(metadata,on=\'document_id\',how=\'left\',validate=\'one_to_one\')\n        verified=merged.type_verified.astype(str).str.lower().isin([\'true\',\'yes\',\'1\'])\n        known=merged.publication_type.fillna(\'\').astype(str).str.lower().str.strip()\n        table(merged.assign(verified_for_analysis=verified),out/\'tables\'/\'publication_type_audit.csv\')\n        descriptive=pd.DataFrame(theta,columns=[f\'T{i+1:02d}\' for i in range(k)])\n        descriptive[\'publication_type\']=known.where(verified,\'unverified\')\n        table(descriptive.groupby(\'publication_type\').mean().reset_index(),out/\'tables\'/\'prevalence_by_publication_type.csv\')\n        if verified.all() and not known.isin([\'\',\'unknown\']).any():\n            is_review=known.str.contains(r\'review|meta.analysis\',regex=True)\n            subset=[d for d,keep in zip(docs,~is_review) if keep]\n            if len(subset)>=max(12,2*k) and len(subset)<len(docs):\n                variants.append((\'verified_nonreview_documents\',cfg,subset))\n            else: pending.append(\'Review-exclusion refit unavailable: no review group or too few verified non-review documents.\')\n        else: pending.append(\'Publication-type sensitivity awaits complete verified classifications; unknown types are never guessed.\')\n    else: pending.append(\'Publication-type sensitivity awaits metadata_csv. Complete metadata_template.csv and verify the classifications.\')\n    if cfg.run_sensitivity:\n        for name,variant,subset in variants:\n            vd=make_units(subset,rep,variant); X=vd[\'X\'][vd[\'training_indices\']]\n            subids=[d[\'id\'] for d in subset]; baseline=theta[[ids.index(i) for i in subids]]\n            for seed in cfg.seeds[:3]:\n                print(f\'Sensitivity: {name} | K={k} | seed={seed}\',flush=True)\n                model=fit_model(X,k,seed,variant,out/\'private\'/\'cache\'/f\'sensitivity_{name}_s{seed}.joblib\')\n                rr,cc,js,jacc=match_topics(P,phi(model))\n                other=aggregate_documents(model.transform(vd[\'X\']),vd[\'units\'],subids)[:,cc]\n                rows.append({\'variant\':name,\'seed\':seed,\'documents\':len(subset),\'training_units\':X.shape[0],\n                    \'iterations\':model.actual_iterations_,\'converged\':model.converged_,\n                    \'matched_js_to_reference\':float(js.mean()),\'matched_top10_jaccard\':float(jacc.mean()),\n                    \'document_topic_l1_mean\':float(np.abs(other-baseline).sum(1).mean()),\n                    \'max_prevalence_change_percentage_points\':float(np.max(np.abs(other.mean(0)-baseline.mean(0)))*100),\n                    \'alpha\':model.doc_topic_prior_,\'eta\':model.topic_word_prior_,\n                    \'comparison\':\'Same normalized vocabulary; model structure and aligned prevalence on the same subset; no cross-unit ELBO comparison.\'})\n                models.append(model)\n        if models:\n            coherence=batch_coherence(models,data[\'texts\'],rep[\'dictionary\'],cfg.coherence_topn)\n            for row,coh in zip(rows,coherence): row.update(c_v_full_corpus_reference=coh[\'c_v\'],c_npmi_full_corpus_reference=coh[\'c_npmi\'])\n    else: pending.append(\'Configuration disabled structural sensitivity refits.\')\n    table(rows,out/\'tables\'/\'sensitivity_runs.csv\')\n    if rows:\n        numeric=pd.DataFrame(rows).select_dtypes(include=\'number\').columns.tolist()\n        numeric=[x for x in numeric if x!=\'seed\']\n        table(pd.DataFrame(rows).groupby(\'variant\')[numeric].agg([\'mean\',\'std\']).pipe(lambda x:x.set_axis([\'_\'.join(c) for c in x.columns],axis=1)).reset_index(),out/\'tables\'/\'sensitivity_summary.csv\')\n    # Fixed-model influence only; do not call this a leave-one-out refitted model.\n    influence=[]\n    for i,d in enumerate(docs):\n        delta=(theta.sum(0)-theta[i])/(len(docs)-1)-theta.mean(0)\n        influence.append({\'document_id\':d[\'id\'],\'filename\':d[\'filename\'],\n            \'max_change_percentage_points\':100*float(np.max(np.abs(delta))),\n            \'definition\':\'Remove one document from the average with topic model fixed; no refit.\'})\n    table(influence,out/\'tables\'/\'fixed_model_document_influence.csv\')\n    write_json(out/\'tables\'/\'sensitivity_status.json\',{\'completed_variants\':sorted(set(r[\'variant\'] for r in rows)),\n        \'pending\':pending,\'interpretation\':\'Sensitivity measures dependence on specifications; it does not prove substantive validity.\'})\n\n\ndef human_validation_pack(docs,rep,data,families,cfg,out):\n    rng=np.random.default_rng(cfg.split_seed); tasks=[]; keys=[]\n    ids=[d[\'id\'] for d in docs]\n    for k,fam in families.items():\n        model=fam[\'model\']; P=phi(model); ut=model.transform(data[\'X\'])\n        dt=aggregate_documents(ut,data[\'units\'],ids)\n        summary,keywords=topic_word_tables(model,rep,dt.mean(0),cfg)\n        for t in range(k):\n            raw=top_terms(model,rep[\'dictionary\'],10)[t]\n            intruders=[w for j,terms in enumerate(top_terms(model,rep[\'dictionary\'],30)) if j!=t for w in terms\n                       if w not in raw and w not in DISPLAY_STOPWORDS and P[t,rep[\'dictionary\'].token2id[w]]<np.median(P[:,rep[\'dictionary\'].token2id[w]])]\n            if not intruders: raise ValueError(\'Could not construct an unambiguous candidate intruder pool; inspect topic overlap.\')\n            intruder=intruders[int(rng.integers(len(intruders)))]\n            genuine=[w for w in raw if w not in DISPLAY_STOPWORDS][:3]\n            if len(genuine)<3: genuine=raw[:3]\n            options=genuine+[intruder]; rng.shuffle(options)\n            answer=\'ABCD\'[options.index(intruder)]\n            doc_ix=np.argsort(dt[:,t])[::-1][:3]\n            references=[]; passages=[]\n            for di in doc_ix:\n                doc=docs[di]; ix=[j for j,u in enumerate(data[\'units\']) if u[\'document_id\']==doc[\'id\']]\n                best=max(ix,key=lambda j:ut[j,t]); u=data[\'units\'][best]\n                references.append(f\'{doc["id"]}: {doc["filename"]}; PDF pp. {u["page_start"]}-{u["page_end"]}\')\n                passages.append(u[\'normalized_excerpt\'])\n            tasks.append({\'raw_top10\':\', \'.join(raw),\n                \'refined_top10\':summary.iloc[t].display_top10,\n                \'representative_sources\':\' | \'.join(references),\'normalized_passages\':\' | \'.join(passages),\n                **{f\'option_{letter}\':word for letter,word in zip(\'ABCD\',options)},\n                \'coherence_rating_1_to_5\':\'\',\'intruder_choice_A_to_D\':\'\',\'proposed_label\':\'\',\'label_evidence_and_page_notes\':\'\',\n                \'financial_crime_claim_supported_yes_no_unclear\':\'\',\'rater_name_or_code\':\'\'})\n            keys.append({\'k\':k,\'topic\':f\'T{t+1:02d}\',\'model_seed\':model.random_state,\'intruder_correct_option\':answer})\n    order=rng.permutation(len(tasks)); blinded=[]; answer_key=[]\n    for i,index in enumerate(order,1):\n        task_id=f\'H{i:03d}\'; blinded.append({\'task_id\':task_id,**tasks[index]}); answer_key.append({\'task_id\':task_id,**keys[index]})\n    directory=out/\'private\'/\'human_validation\'\n    table(blinded,directory/\'rater_1.csv\'); table(blinded,directory/\'rater_2.csv\'); table(answer_key,directory/\'answer_key.csv\')\n    table([{\'k\':k,\'topic\':row.topic,\'model_seed\':fam[\'model\'].random_state,\'raw_keywords\':row.raw_top10,\n        \'suggested_keyword_label\':row.suggested_label,\'final_human_label\':\'\',\'coder_1_label\':\'\',\'coder_2_label\':\'\',\n        \'adjudication_notes\':\'\',\'supporting_document_ids_and_pdf_pages\':\'\',\'claim_scope_and_limitations\':\'\'}\n        for k,fam in families.items()\n        for row in topic_word_tables(fam[\'model\'],rep,aggregate_documents(fam[\'model\'].transform(data[\'X\']),data[\'units\'],ids).mean(0),cfg)[0].itertuples()],directory/\'topic_label_audit.csv\')\n    instructions=\'\'\'# Independent human validation (pending)\n\nGive rater_1.csv and rater_2.csv to two independent coders. Keep answer_key.csv\nand suggested model-selection results with the coordinator until coding ends.\nDo not show one coder the other\'s decisions. Tasks are randomized and K/topic IDs\nare hidden; keywords can still reveal similarity, so this is partial blinding.\n\nFor every row: rate semantic coherence from 1 (incoherent) to 5 (very coherent),\nchoose the least-fitting item from A-D, propose a concise label, and record PDF\npages supporting that label and any proposed financial-crime claim. The supplied\npassages are normalized tokens, NOT verbatim quotations; inspect the actual PDFs.\nThe algorithmically inserted intruder is a designed task answer, not objective\nground truth about disciplinary meaning. Review ambiguous tasks before coding\nand preserve any exclusions with reasons. Do not change a key after seeing ratings.\n\nAfter independent coding, call analyze_human_ratings(...) in the notebook. It\ncomputes ordinal weighted Cohen kappa, exact agreement, intrusion accuracy and\nnominal choice agreement. Small task counts and rater selection limit inference.\nComplete topic_label_audit.csv separately, retaining disagreements and adjudication.\nDo not report validation as completed just because these blank files were created.\n\'\'\'\n    (directory/\'INSTRUCTIONS.md\').write_text(instructions,encoding=\'utf-8\')\n    write_json(out/\'tables\'/\'human_validation_status.json\',{\'status\':\'pending\',\'tasks\':len(tasks),\n        \'required\':\'Two independent completed rating files and documented label adjudication\',\n        \'ratings_computed\':False})\n\n\ndef analyze_human_ratings(rater1_csv,rater2_csv,answer_key_csv,output_dir):\n    a=pd.read_csv(rater1_csv).fillna(\'\'); b=pd.read_csv(rater2_csv).fillna(\'\'); key=pd.read_csv(answer_key_csv)\n    for d in (a,b,key):\n        if d.task_id.duplicated().any(): raise ValueError(\'Task IDs must be unique.\')\n    if set(a.task_id)!=set(key.task_id) or set(b.task_id)!=set(key.task_id): raise ValueError(\'Both raters must have exactly the keyed task IDs.\')\n    merged=key.merge(a,on=\'task_id\',validate=\'one_to_one\').merge(b,on=\'task_id\',suffixes=(\'_r1\',\'_r2\'),validate=\'one_to_one\')\n    c1=pd.to_numeric(merged.coherence_rating_1_to_5_r1,errors=\'coerce\')\n    c2=pd.to_numeric(merged.coherence_rating_1_to_5_r2,errors=\'coerce\')\n    q1=merged.intruder_choice_A_to_D_r1.astype(str).str.upper().str.strip()\n    q2=merged.intruder_choice_A_to_D_r2.astype(str).str.upper().str.strip()\n    complete=c1.isin([1,2,3,4,5])&c2.isin([1,2,3,4,5])&q1.isin(list(\'ABCD\'))&q2.isin(list(\'ABCD\'))\n    if not complete.all(): raise ValueError(f\'{int((~complete).sum())} tasks have missing/invalid ratings. Finish independent coding; missing answers are not imputed.\')\n    def metrics(mask,name):\n        x=c1[mask]; y=c2[mask]; p=q1[mask]; q=q2[mask]; answers=merged.loc[mask,\'intruder_correct_option\']\n        if len(x)<2: raise ValueError(\'Too few tasks for agreement analysis.\')\n        return {\'scope\':name,\'tasks\':len(x),\'coherence_exact_agreement\':float((x==y).mean()),\n            \'coherence_quadratic_weighted_kappa\':float(cohen_kappa_score(x,y,labels=[1,2,3,4,5],weights=\'quadratic\')),\n            \'intrusion_choice_exact_agreement\':float((p==q).mean()),\n            \'intrusion_choice_kappa\':float(cohen_kappa_score(p,q,labels=list(\'ABCD\'))),\n            \'rater1_intrusion_accuracy\':float((p==answers).mean()),\'rater2_intrusion_accuracy\':float((q==answers).mean()),\n            \'rater1_mean_coherence\':float(x.mean()),\'rater2_mean_coherence\':float(y.mean())}\n    results=[metrics(np.ones(len(merged),dtype=bool),\'all_tasks\')]\n    results.extend(metrics(merged.k==k,f\'K={k}\') for k in sorted(merged.k.unique()))\n    out=Path(output_dir); result=table(results,out/\'human_agreement_results.csv\')\n    table(merged,out/\'private\'/\'merged_human_ratings.csv\')\n    labels_done=all(merged[f\'proposed_label_{r}\'].astype(str).str.strip().ne(\'\').all() and\n                    merged[f\'label_evidence_and_page_notes_{r}\'].astype(str).str.strip().ne(\'\').all() for r in (\'r1\',\'r2\'))\n    write_json(out/\'human_validation_status.json\',{\'rating_status\':\'completed\',\'label_evidence_fields_complete\':bool(labels_done),\n        \'adjudication_status\':\'Requires human review of topic_label_audit.csv\',\n        \'warning\':\'Kappa can be undefined for constant ratings; a NaN is not evidence of perfect agreement.\'})\n    return result\n\n\nREVIEWER_CROSSWALK = [\n (\'E7; R1.4g; R1.5c; R1.A8; R1.A17\',\'K choice and random-seed stability\',\'model_selection_summary.csv; seed_stability_cv.csv; full_corpus_seed_stability.csv; seed_topic_alignment.csv\',\'Computed after research run; human interpretability remains pending\'),\n (\'R1.5b; R1.A17\',\'Convergence and 250 iterations\',\'convergence_traces_cv.csv; convergence_traces_full_corpus.csv; final_convergence_trace.*\',\'Actual outer updates and inner cap distinguished; nonconvergence explicitly reported\'),\n (\'R1.4c; R1.A4\',\'Exact preprocessing, duplicate, reference and chunk rules\',\'corpus_manifest.csv; extraction_page_audit.csv; normalization_map.csv; unit_manifest.csv; run_config.json\',\'Rules specified; extraction flags and publication identity require review\'),\n (\'R1.5d; R1.A6\',\'Document-normalized prevalence; fewer than 50 chunks\',\'document_chunk_counts.csv; topic_prevalence.csv; document_topic_matrix.csv; sensitivity_runs.csv\',\'All short documents retained without resampling; all chunks used for inference\'),\n (\'R1.A7; E7\',\'Author labels and missing Appendix B\',\'topic_summary.csv; representative_documents.csv; private/human_validation/topic_label_audit.csv\',\'Provisional keyword labels only; two-coder support and adjudication pending\'),\n (\'R1.A9\',\'Co-occurrence graph is descriptive\',\'keyword_network_definition.json; keyword_cooccurrence_network.*; keyword_cooccurrence.graphml\',\'Explicitly not independent confirmation of LDA\'),\n (\'E5\',\'Mixing primary studies and reviews\',\'metadata_template.csv; prevalence_by_publication_type.csv and verified_nonreview_documents sensitivity when classifications supplied\',\'Author-verified publication types needed; overlapping evidence cannot be inferred from word counts\'),\n (\'E6; R2.4b\',\'Study quality/risk of bias\',\'metadata_template.csv quality-appraisal fields\',\'Human, design-appropriate study-quality appraisal remains outside LDA\'),\n (\'E9; R1.A5\',\'Occurrence counts are not evidence strength\',\'all_normalized_keywords.csv; top10_normalized_keywords_overall.csv; topic_prevalence.csv\',\'Explicit denominators; no claim of crime incidence or independent evidence counts\'),\n (\'E1e; R1.4e; R2.4d; R1.A17\',\'Code/data distinction and package versions\',\'run_config.json; environment_versions.json; requirements-colab.txt; derived_outputs.zip\',\'Copyrighted PDFs and passage text excluded from derived archive; authors verify sharing permissions\'),\n (\'R1.6b; R1.6c; R1.A13\',\'Readable methods, tables, figures and appendices\',\'METHODS_TEMPLATE.md; topic_summary.csv; figures in PNG/SVG/PDF\',\'Final prose and topic labels require completed research results\'),\n (\'R1.A10; R2.4e\',\'Small corpus; exploratory interpretation\',\'corpus_flow.json; model-selection notes; sensitivity outputs\',\'Independent N is number of PDFs, not chunks; cannot establish comprehensive coverage\'),\n (\'E3; E4; R1.3e; R1.4b; R1.4f; R1.A2; R1.A3; R1.A11\',\'Search, access exclusions, screening and PRISMA\',\'Separate screening/eligibility record\',\'Cannot be repaired by running LDA; folder N is not a replacement PRISMA denominator\'),\n (\'E8; E10; R1.4d; R1.A5; R1.A18; R2.4c\',\'Qwen/Ollama prompts, evidence validation and graph provenance\',\'Separate LLM pipeline and human validation records\',\'This notebook contains no LLM evidence extraction and cannot retrospectively validate that stage\'),\n]\n\n\ndef write_run_report(docs,cfg,out,decision,families):\n    m=families[decision[\'final_k\']][\'model\']\n    table([{\'reviewer_item\':a,\'concern\':b,\'evidence_outputs\':c,\'completion_boundary\':d} for a,b,c,d in REVIEWER_CROSSWALK],out/\'tables\'/\'reviewer_response_crosswalk.csv\')\n    text=f\'\'\'# Analysis run: {VERSION}\n\nProfile: **{cfg.profile}**. Original included PDFs: **{len(docs)}**.\nProvisional diagnostic recommendation: **K={decision[\'recommended_k\']}**.\nFinal fitted K: **{decision[\'final_k\']}**, medoid seed: **{m.random_state}**.\nActual final-model EM updates: **{m.actual_iterations_}**; numerical stopping criterion met: **{m.converged_}**.\n\n{\' \'.join(decision[\'notes\'])}\n\n## Interpretation\n\nTraining score is an approximate variational evidence lower bound, not an exact\nmarginal likelihood. Predictive log likelihood is a plug-in document-completion\nscore: topic proportions are inferred from one random token half and the other\nhalf is scored. It is not an unbiased marginal-likelihood estimator. Vocabulary\nand learned phrases are fitted within each training fold. Validation splits keep\neach PDF and flagged near-duplicate family together. The folds guide model\nselection; these are not an untouched final test set or population estimates.\n\nChunking is performed within each PDF after normalization, with size {cfg.chunk_size},\nzero overlap and tails under {cfg.min_tail} tokens merged into the previous chunk.\nAt most {cfg.max_train_chunks_per_doc} evenly spaced chunks per PDF train the model,\nwithout replacement. Every available chunk is used to infer its PDF\'s topic\ndistribution. Documents with fewer than 50 chunks contribute all their chunks;\nnone is duplicated or excluded for that reason. Longer PDFs can still exert more\ninfluence on fitting. Unit and cap sensitivity checks evaluate this dependence.\n\nFor PDF d, theta_dk = sum_c(n_dc * theta_dck) / sum_c(n_dc), where n_dc counts\nretained vocabulary tokens in nonoverlapping chunk c. Overall prevalence is\n(1/D) * sum_d(theta_dk). Each PDF has equal weight in that final average. This\nmaps thematic attention in the included literature, not financial-crime\nincidence, methodological quality or the strength of independent evidence.\n\nBootstrap intervals resample original PDFs with the fitted model held fixed.\nThey describe conditional dispersion only, excluding model-selection, seed,\nextraction and search-coverage uncertainty. Seed dispersion is reported separately.\nNetwork edges represent document-level co-presence. They cannot independently\nvalidate LDA, establish causality, or validate author-derived substantive labels.\n\n## Reviewer response status\n\nSee tables/reviewer_response_crosswalk.csv and the [live reviewer document]({REVIEW_URL}).\nBlank coding templates are not completed human validation. Publication-type\nsensitivity needs verified metadata. Study-quality appraisal, screening reliability,\nPRISMA reconciliation and Qwen validation remain separate author tasks.\n\n## Navigation\n\n- tables/model_selection_summary.csv: cross-validation and multi-seed comparison.\n- model_selection_decision.json: transparent decision rule and qualifications.\n- tables/final_goodness_of_fit.csv: final fit, coherence and convergence.\n- tables/topic_summary.csv: concise provisional topic table.\n- tables/topic_prevalence.csv: equal-document estimates and weighting alternatives.\n- lda_interactive.html: interactive pyLDAvis (token-weighted areas; topic IDs preserved).\n- figures/: all figures in 300-dpi PNG plus vector SVG/PDF.\n- private/: cached models, text, excerpts and human-validation worksheets.\n- derived_outputs.zip: tables, figures and run reports; excludes private content/PDFs.\n\n## Sources for the implementation\n\n- [LDA and variational inference](https://jmlr.org/papers/v3/blei03a.html)\n- [scikit-learn LDA API](https://scikit-learn.org/1.7/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html)\n- [Gensim coherence pipeline](https://radimrehurek.com/gensim/models/coherencemodel.html)\n- [Wallach and colleagues on evaluating topic models](https://homepages.inf.ed.ac.uk/imurray2/pub/09etm/)\n- [Röder and colleagues on topic coherence](https://doi.org/10.1145/2684822.2685324)\n\'\'\'\n    (out/\'RUN_REPORT.md\').write_text(text,encoding=\'utf-8\')\n    method=f\'\'\'# Methods text to adapt after validation\n\n**Draft with computational facts only; add final human labels and author decisions.**\n\nWe conducted an exploratory LDA analysis of {len(docs)} full-text documents.\nPDF text was extracted locally using PyMuPDF, with a pypdf fallback and logged\npage-level diagnostics. Delivery covers and a hash-verified chapter-front-matter\nrange were excluded. Reference sections were removed only at standalone\nreference headings after {cfg.reference_min_fraction:.0%} of cleaned text. Exact\nfile/content duplicates were logged; near-duplicate candidates were retained and\nplaced within the same validation fold. Text normalization included Unicode\nrepair, an explicit domain-alias dictionary, noun lemmatization and auditable\nadjacent-repeat removal. Learned phrase detection and vocabulary filters were\nfitted on training documents within each validation fold.\n\nWe compared K={list(cfg.k_values)} with seeds {list(cfg.seeds)} using\n{cfg.folds} document-grouped folds. LDA used batch variational inference,\nalpha={cfg.alpha if cfg.alpha is not None else \'1/K\'} and eta={cfg.eta}, a maximum\nof {cfg.max_iter} outer EM updates and {cfg.max_doc_update_iter} inner updates.\nWe logged the variational bound at every outer update. Training perplexity,\ndocument-completion log scores, C_v (110-token window), C_NPMI (10-token window),\ntop-{cfg.coherence_topn} diversity, and Hungarian-matched seed stability informed\nmodel selection. Complete results and the declared selection rule accompany\nthis script. Fold dispersion is descriptive; the validation folds were used for\nselection and are not an independent post-selection test set.\n\nThe computational rule recommended K={decision[\'recommended_k\']}; the final\nfit used K={decision[\'final_k\']} and the medoid seed {m.random_state}.\n**Before reporting this as the preferred solution, resolve diagnostic flags,\ncomplete independent interpretability assessments, and state any override.**\nTopic labels are author interpretations that require supporting document/page\nreferences. Prevalence averages token-weighted chunk proportions within each\ndocument, then weights documents equally. The keyword network describes\ndocument-level lexical co-occurrence and is not an independent model-validation\ntest. Results describe this corpus\'s thematic structure; they do not estimate\ncrime prevalence, validate proposed safeguards, or substitute for study appraisal.\n\n**Complete separately:** screening/search/PRISMA account; study-type/quality\nappraisal; coder identities and blinding; agreement results and disagreements;\nfinal K justification; label adjudication; limitations; data-sharing declaration.\n\'\'\'\n    (out/\'METHODS_TEMPLATE.md\').write_text(method,encoding=\'utf-8\')\n\n\ndef validate_outputs(out,docs,data,theta,decision):\n    checks={}\n    checks[\'all_document_rows_present\']=theta.shape[0]==len(docs)\n    checks[\'document_probabilities_sum_to_one\']=bool(np.allclose(theta.sum(1),1))\n    checks[\'mean_prevalence_sums_to_one\']=bool(np.isclose(theta.mean(0).sum(),1))\n    checks[\'nonnegative_counts\']=bool((data[\'X\'].data>=0).all())\n    checks[\'training_units_no_replacement\']=len(data[\'training_indices\'])==len(set(data[\'training_indices\']))\n    s=pd.read_csv(out/\'tables\'/\'document_validation_splits.csv\')\n    checks[\'validation_has_no_group_leakage\']=all(not set(g[g.part==\'train\'].group)&set(g[g.part==\'validation\'].group) for _,g in s.groupby(\'fold\'))\n    checks[\'every_document_validated_once\']=bool((s[s.part==\'validation\'].document_id.value_counts()==1).all())\n    p=pd.read_csv(out/\'tables\'/\'topic_word_probabilities.csv\')\n    checks[\'topic_word_probabilities_sum_to_one\']=bool(np.allclose(p.drop(columns=\'term\').sum(),1))\n    top=pd.read_csv(out/\'tables\'/\'topic_top10_normalized_keywords.csv\')\n    checks[\'display_keyword_weights_sum_to_one\']=bool(np.allclose(top.groupby(\'topic\').normalized_weight_within_display_top10.sum(),1))\n    for method in (\'cosine\',\'js\'):\n        sim=pd.read_csv(out/\'tables\'/f\'topic_similarity_{method}.csv\').drop(columns=\'topic\').to_numpy()\n        checks[f\'{method}_similarity_symmetric\']=bool(np.allclose(sim,sim.T))\n    required=[\'topic_summary.csv\',\'final_goodness_of_fit.csv\',\'top10_normalized_keywords_overall.csv\',\n        \'topic_prevalence.csv\',\'model_selection_summary.csv\',\'convergence_traces_full_corpus.csv\',\n        \'keyword_network_display_edges.csv\',\'reviewer_response_crosswalk.csv\']\n    checks[\'requested_tables_exist\']=all((out/\'tables\'/p).exists() for p in required)\n    checks[\'interactive_lda_created\']=(out/\'lda_interactive.html\').exists()\n    write_json(out/\'verification.json\',{\'checks\':checks,\'all_pass\':all(checks.values()),\n        \'validation_scope\':\'Software consistency, not substantive topic validity\',\'profile\':decision.get(\'profile\',\'see run_config.json\')})\n    if not all(checks.values()): raise AssertionError(f\'Output integrity check failed: {checks}\')\n    return checks\n\n\ndef package_outputs(out):\n    out=Path(out); manifest=[]\n    files=[p for p in out.rglob(\'*\') if p.is_file() and \'private\' not in p.relative_to(out).parts and p.suffix.lower()!=\'.zip\' and p.name!=\'output_checksums.csv\']\n    for p in sorted(files):\n        manifest.append({\'file\':p.relative_to(out).as_posix(),\'bytes\':p.stat().st_size,\'sha256\':hashlib.sha256(p.read_bytes()).hexdigest()})\n    table(manifest,out/\'output_checksums.csv\'); files.append(out/\'output_checksums.csv\')\n    with zipfile.ZipFile(out/\'derived_outputs.zip\',\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n        for p in files: z.write(p,p.relative_to(out).as_posix())\n\n\ndef run_pipeline(config=None):\n    cfg=(config or Config()).checked()\n    source=Path(cfg.input_dir).expanduser().resolve()\n    pdfs=sorted(p for p in source.rglob(\'*\') if p.is_file() and p.suffix.lower()==\'.pdf\')\n    if not pdfs: raise FileNotFoundError(f\'No PDFs found at {source}\')\n    fingerprints=[(p.relative_to(source).as_posix(),hashlib.sha256(p.read_bytes()).hexdigest()) for p in pdfs]\n    versions={p:im.version(p) for p in PACKAGES}\n    code_hash=hashlib.sha256(Path(__file__).read_bytes()).hexdigest() if \'__file__\' in globals() else globals().get(\'NOTEBOOK_CODE_SHA256\',VERSION)\n    key=digest({\'config\':asdict(cfg),\'pdfs\':fingerprints,\'versions\':versions,\'code\':code_hash,\n        \'aliases\':DOMAIN_ALIASES,\'page_rules\':load_page_rules(cfg),\n        \'metadata_sha\':hashlib.sha256(Path(cfg.metadata_csv).read_bytes()).hexdigest() if cfg.metadata_csv else \'\'})\n    out=Path(cfg.output_dir).expanduser().resolve()/(\'run_\'+key[:12])\n    for name in (\'tables\',\'figures\',\'private\'): (out/name).mkdir(parents=True,exist_ok=True)\n    public_config=asdict(cfg)\n    for field in (\'input_dir\',\'output_dir\'): public_config[field]=\'USER_SUPPLIED_DIRECTORY\'\n    for field in (\'metadata_csv\',\'page_rules_csv\'): public_config[field]=Path(public_config[field]).name if public_config[field] else \'\'\n    write_json(out/\'run_config.json\',public_config)\n    write_json(out/\'private\'/\'runtime_config.json\',asdict(cfg))\n    write_json(out/\'tables\'/\'preprocessing_definitions.json\',{\'domain_aliases\':DOMAIN_ALIASES,\n        \'model_stopwords\':sorted(set(ENGLISH_STOP_WORDS)|MODEL_BOILERPLATE|set(cfg.extra_stopwords)),\n        \'display_stopwords\':sorted(DISPLAY_STOPWORDS),\n        \'note\':\'Display filtering does not alter model fitting or raw-term coherence. Paragraph block extraction is approximate; inspect source-page references.\'})\n    write_json(out/\'environment_versions.json\',{\'python\':sys.version,\'platform\':platform.platform(),\'packages\':versions,\'pipeline_version\':VERSION,\'code_sha256\':code_hash,\'run_hash\':key})\n    if \'__file__\' in globals():\n        (out/\'analysis_source.py\').write_text(Path(__file__).read_text(encoding=\'utf-8\'),encoding=\'utf-8\')\n    resolved=sorted({f\'{d.metadata["Name"]}=={d.version}\' for d in im.distributions() if d.metadata.get(\'Name\')})\n    (out/\'requirements-resolved.txt\').write_text(\'\\n\'.join(resolved)+\'\\n\',encoding=\'utf-8\')\n    print(f\'Outputs/checkpoints: {out}\',flush=True)\n    setup_resources(out,cfg)\n    docs,manifest=audit_corpus(cfg,out)\n    summary,runs=cross_validate(docs,cfg,out)\n    decision=recommend_k(summary,cfg,out)\n    print(f\'Diagnostic K={decision["recommended_k"]}; final K={decision["final_k"]}. Check all qualifications.\',flush=True)\n    rep,data,families=fit_final_candidates(docs,cfg,out,decision)\n    theta,topics,keywords=export_topics(docs,rep,data,families,cfg,out,decision)\n    keyword_network(docs,rep,data,cfg,out)\n    sensitivity_analysis(docs,rep,data,families,cfg,out,decision,theta)\n    human_validation_pack(docs,rep,data,families,cfg,out)\n    write_run_report(docs,cfg,out,decision,families)\n    validate_outputs(out,docs,data,theta,decision)\n    package_outputs(out)\n    print(f\'COMPLETE: {out}\\nHuman labels/validation remain pending. Profile: {cfg.profile}\',flush=True)\n    return {\'output_dir\':out,\'decision\':decision,\'topic_summary\':topics,\'model_selection\':summary,\'documents\':len(docs)}\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--input-dir\',required=True)\n    parser.add_argument(\'--output-dir\',default=\'lda_results\')\n    parser.add_argument(\'--profile\',choices=[\'research\',\'smoke\'],default=\'research\')\n    parser.add_argument(\'--metadata-csv\',default=\'\')\n    parser.add_argument(\'--page-rules-csv\',default=\'\')\n    parser.add_argument(\'--config-json\',help=\'Optional JSON with other Config fields; CLI paths/profile take precedence\')\n    args=parser.parse_args(); settings=json.loads(Path(args.config_json).read_text(encoding=\'utf-8\')) if args.config_json else {}\n    settings.update(input_dir=args.input_dir,output_dir=args.output_dir,profile=args.profile,\n                    metadata_csv=args.metadata_csv,page_rules_csv=args.page_rules_csv)\n    run_pipeline(Config(**settings))\n\n\nif __name__==\'__main__\':\n    main()\n'
REQUIREMENTS = 'Jinja2==3.1.6\nMarkupSafe==3.0.3\nPyMuPDF==1.26.4\nclick==8.5.0\ncloudpickle==3.1.2\ncontourpy==1.4.0\ncycler==0.12.1\nfonttools==4.65.0\nfuncy==2.1\ngensim==4.4.0\njoblib==1.6.0\nkiwisolver==1.5.1\nmatplotlib==3.10.6\nnetworkx==3.5\nnltk==3.9.2\nnumexpr==2.14.2\nnumpy==2.2.6\npackaging==26.3\npandas==2.2.3\npillow==12.3.0\npip==26.2.1\npyLDAvis==3.4.1\npyparsing==3.3.2\npypdf==6.0.0\npython-dateutil==2.9.0.post0\npytz==2026.3.post1\nregex==2026.9.10\nscikit-learn==1.7.2\nscipy==1.15.3\nsetuptools==84.0.0\nsix==1.17.0\nsmart_open==8.0.1\nthreadpoolctl==3.6.0\ntqdm==4.70.1\ntzdata==2026.4\nwrapt==2.4.1\n'
CONFIG = json.loads('{\n  "input_dir": "USER_SUPPLIED_DIRECTORY",\n  "output_dir": "USER_SUPPLIED_DIRECTORY",\n  "profile": "research",\n  "k_values": [\n    2,\n    3,\n    4,\n    5,\n    6,\n    7,\n    8,\n    9,\n    10,\n    11,\n    12\n  ],\n  "seeds": [\n    11,\n    42,\n    97,\n    2026,\n    31415\n  ],\n  "folds": 3,\n  "split_seed": 20260915,\n  "max_iter": 250,\n  "max_doc_update_iter": 250,\n  "perp_tol": 0.1,\n  "mean_change_tol": 0.001,\n  "alpha": null,\n  "eta": 0.01,\n  "unit": "chunk",\n  "chunk_size": 400,\n  "min_tail": 100,\n  "max_train_chunks_per_doc": 50,\n  "min_document_tokens": 100,\n  "min_df": 2,\n  "max_df": 1.0,\n  "max_features": 10000,\n  "learned_phrases": true,\n  "phrase_min_count": 10,\n  "phrase_threshold": 10.0,\n  "lemmatize": true,\n  "strip_references": true,\n  "reference_min_fraction": 0.4,\n  "near_duplicate_threshold": 0.93,\n  "coherence_topn": 10,\n  "coherence_tolerance": 0.02,\n  "minimum_stability": 0.7,\n  "minimum_diversity": 0.6,\n  "minimum_converged_fraction": 0.8,\n  "final_k_override": null,\n  "final_k_reason": "",\n  "bootstrap_replicates": 2000,\n  "network_terms": 40,\n  "network_min_docs": 3,\n  "network_min_jaccard": 0.05,\n  "network_min_npmi": 0.05,\n  "network_max_df": 0.9,\n  "network_max_edges": 150,\n  "run_sensitivity": true,\n  "enable_ocr": false,\n  "ocr_language": "eng",\n  "metadata_csv": "",\n  "page_rules_csv": "",\n  "extra_stopwords": []\n}')
(WORK / "Metaverse_LDA_Colab.py").write_text(SOURCE, encoding="utf-8")
(WORK / "requirements-resolved.txt").write_text(REQUIREMENTS, encoding="utf-8")
(WORK / "run_config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print("Python:", sys.version)
print("Prepared source/configuration in", WORK)


## Install dependencies
The recorded environment was Python 3.13.15 on Linux. This cell installs the recorded packages; record any environment differences. WordNet is obtained by the pipeline if absent.


In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(WORK / "requirements-resolved.txt")])


## Set the input and output directories
Place the 65 original PDFs in a directory accessible to this runtime. In Colab, mount Drive separately if using a Drive folder. Preserve filenames and compare file hashes against corpus_manifest.csv. Replace the two values below.


In [ ]:
INPUT_DIR = Path("/path/to/the/65_PDFs")
OUTPUT_DIR = Path("/path/to/lda_results")
if not INPUT_DIR.is_dir():
    raise FileNotFoundError("Set INPUT_DIR to the actual corpus directory before running.")
print("PDF files:", len(list(INPUT_DIR.rglob("*.pdf"))))


## Run the research profile
This is the substantial fitting step. It performs model selection and sensitivity checks with checkpointed fits. Do not interpret a later smoke-profile run as the reported research result.


In [ ]:
command = [sys.executable, str(WORK / "Metaverse_LDA_Colab.py"), "--input-dir", str(INPUT_DIR), "--output-dir", str(OUTPUT_DIR), "--profile", "research", "--config-json", str(WORK / "run_config.json")]
subprocess.run(command, check=True)


## Inspect outputs
The selected output directory contains run_<fingerprint> folders, with tables and figures. Read model_selection_decision.json and validation status before interpreting the topic labels. For the same-seed sensitivity table, use the supplied audit_paired_sensitivity.py against the saved run. No external archive helper is required.


In [ ]:
for run in sorted(OUTPUT_DIR.glob("run_*")):
    decision = run / "model_selection_decision.json"
    if decision.exists():
        print(run)
        print(decision.read_text(encoding="utf-8"))
